# DO2 Multiclass Status Prediction — Time-Safe Revision

This notebook predicts `C`, `I`, `MR`, `N`, `P`, and `V` using entry-time
features and verified N/I/MR rules.

Important corrections in this revision:

- train, validation, and test are split by `created_at`;
- every split comes from `do2`; training covers 2022-01-01 through
  2024-12-31, validation covers January–June 2025, and test covers
  July–December 2025;
- split end dates are exclusive and are written as the first day of
  the next period;
- all eligible non-V training rows and a deterministic V sample come
  from the 2022–2024 training period;
- validation/test rows are never removed using their target status;
- employee and history facts are censored to information available at entry;
- duplicated feature-function definitions have been removed;
- identical model-frame preparation is used for train/validation/test;
- id uniqueness and split overlap are mandatory checks.

Run using **Restart Kernel → Run All**. Do not trust partial, out-of-order runs.

## 1. Setup

In [1]:
import gc
import importlib.metadata
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sqlalchemy import text

cwd = Path.cwd().resolve()
project_candidates = [cwd, cwd.parent, cwd / "do2_status_prediction"]
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / "pyproject.toml").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Run this notebook from the do2_status_prediction project root or its src directory."
    )

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from do2_model.config import Settings
from do2_model.database import get_engine

settings = Settings()
if not settings.database_url:
    raise RuntimeError(f"DATABASE_URL is missing from {PROJECT_ROOT / '.env'}")

engine = get_engine(settings)

print(f"Project: {PROJECT_ROOT}")
print(f"Database schema: {settings.db_schema}")
print("DATABASE_URL loaded from .env (value hidden).")


Project: /home/ruthikreddy/Desktop/do2/do2_status_prediction
Database schema: public
DATABASE_URL loaded from .env (value hidden).


## 2. Parameters and temporal split policy

In [2]:
# PostgreSQL source. Every split is selected from the same source table.
TRAIN_TABLE = "do2"
SOURCE_TABLE = "do2"

# Use the timestamp at which a record became available to the model.
SPLIT_DATE_COLUMN = "created_at"

TARGET_MODE = "reduced_lifecycle_risk"

CORE_OUTCOME_STATUSES = ("I", "MR", "N", "V")

SNAPSHOT_STATUSES = ("C", "I", "MR", "N", "P", "V")

LOAD_STATUSES = SNAPSHOT_STATUSES

if TARGET_MODE == "reduced_lifecycle_risk":

    VALID_STATUSES = CORE_OUTCOME_STATUSES

elif TARGET_MODE == "six_status_snapshot":

    VALID_STATUSES = SNAPSHOT_STATUSES

else:

    raise ValueError(f"Unknown TARGET_MODE: {TARGET_MODE}")

VALID_DO2_GROUPS = ("IA", "IB", "IIA", "IIB", "ID")

TRAIN_START = "2022-01-01"
TRAIN_END = "2025-01-01"
VALIDATION_START = "2025-01-01"
VALIDATION_END = "2025-07-01"
TEST_START = "2025-07-01"
TEST_END = "2026-01-01"

# Keep every eligible non-V training row and a deterministic V sample.
# Reduce this value if memory is limited.
MAX_TRAIN_V_ROWS = 250_000

# Validation and test remain natural-distribution samples.
# Use None only when the machine can hold the complete split.
MAX_VALIDATION_ROWS = 250_000
MAX_TEST_ROWS = 250_000

# Cache expensive enriched query results. Set REFRESH_DATA_CACHE=True only when
# source data or enrichment SQL changes.
USE_DATA_CACHE = True
REFRESH_DATA_CACHE = False
DATA_CACHE_VERSION = "time_safe_v6_status_independent_history"
DATA_CACHE_DIR = PROJECT_ROOT / "data" / "model_cache"

# This is allowed only for known bad historical training labels. Validation and
# test never use the target status to remove rows.
DROP_INCOMPLETE_V_TRAINING_LABELS = True

# If a row's created_at cannot be parsed (NULL, malformed text, or a
# PostgreSQL "infinity"/"-infinity" sentinel timestamp) it cannot be
# time-split safely. These rows are excluded before splitting and the
# excluded count is reported, instead of silently becoming NaT and
# failing the split_validation check later.
DROP_UNPARSEABLE_CREATED_AT_ROWS = True

# Softly correct the deliberately sampled training distribution toward the
# natural class distribution in the training period.
# Swept automatically in section 8 against validation macro-F1 instead of
# being fixed. This is only the fallback/default if the sweep is skipped.
PRIOR_CORRECTION_POWER = 0.5
PRIOR_CORRECTION_POWER_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)


# Optional, bounded feature-group ablation. It is valuable for proving that

# each rule family adds validation-month performance, but it trains extra models.

RUN_FEATURE_GROUP_ABLATION = True

ABLATION_ITERATIONS = 400

# appendix_j is a current snapshot, not a versioned rule table. Leave its
# historical rule features disabled until a rule effective-date/version is
# available or the business confirms the rules were unchanged for 2022-2025.
USE_CURRENT_APPENDIX_J_FOR_HISTORICAL_ROWS = False


# Model settings. Change TASK_TYPE to "CPU" if GPU training is unavailable.
TASK_TYPE = "GPU"
DEVICES = "0"
ITERATIONS = 2_000
LEARNING_RATE = 0.03
DEPTH = 6
EARLY_STOPPING_ROUNDS = 150
RANDOM_SEED = 42

MODEL_DIR = PROJECT_ROOT / "models_prac_v4_time_safe"
SAVE_ARTIFACTS = True

print("Source:", f"{settings.db_schema}.{SOURCE_TABLE}")

print("Target mode:", TARGET_MODE)

print("Model statuses:", VALID_STATUSES)

if TARGET_MODE == "six_status_snapshot":

    print("WARNING: C and P are mutable lifecycle snapshots, not stable entry outcomes.")

print("Split timestamp:", SPLIT_DATE_COLUMN)
print("Training window:", TRAIN_START, "to", TRAIN_END)
print("Validation window:", VALIDATION_START, "to", VALIDATION_END)
print("Test window:", TEST_START, "to", TEST_END)
print("Maximum sampled V training rows:", MAX_TRAIN_V_ROWS)
print("Validation row cap:", MAX_VALIDATION_ROWS)
print("Test row cap:", MAX_TEST_ROWS)
print("Use enriched-data cache:", USE_DATA_CACHE)


Source: public.do2
Target mode: reduced_lifecycle_risk
Model statuses: ('I', 'MR', 'N', 'V')
Split timestamp: created_at
Training window: 2022-01-01 to 2025-01-01
Validation window: 2025-01-01 to 2025-07-01
Test window: 2025-07-01 to 2026-01-01
Maximum sampled V training rows: 250000
Validation row cap: 250000
Test row cap: 250000
Use enriched-data cache: True


## 3. Load and enrich data

The prediction-time grain is one row per `do2.id`. Lateral history queries are
aggregated to one row per selected DO2. Status-dependent history currently uses the decision timestamp when
available and a `created_at` fallback when it is missing. Section 3
reports the fallback population so it can be replaced with a verified
point-in-time decision timestamp before production use.

The current `employee` table may still be an updated-in-place snapshot. Values
from employee rows created after the DO2 are censored, but a true employee
history table should be used later if one is available.

In [3]:
import time
import pandas as pd

from sqlalchemy import Date, DateTime, inspect, text


LOAD_STATUS_SQL = ", ".join(f"'{status}'" for status in LOAD_STATUSES)

MODEL_STATUS_SQL = ", ".join(f"'{status}'" for status in VALID_STATUSES)


# The application code treats these employee status codes as effective.
EFFECTIVE_EMPLOYEE_STATUS_CODES = ("E", "N", "R", "D", "M", "W", "T")

# Exact lists traced from the Java DO2 processor.
ABOLISHED_DESCRIPTIONS_7CPC = (
    "CILQ", "SNLQ", "FAA", "STRINC", "HCA", "FLCHCA", "ADALLC",
    "ACTGA", "COMNDO", "INSTAL", "OUTFIT", "HSINC", "HCWA",
    "CONDMT", "BOUNTY", "RLCA1", "RLCA2", "RLCA3", "RLCA4",
    "TYOJ", "TYRJ", "DAILY",
)
MUTEX_DESCRIPTIONS = (
    "TYOJ", "TYRJ", "DRDSRA", "AL", "CL", "SL", "EXTNAL", "EXTNCL",
    "EXTNSL", "TJRA", "RTYOJ", "EXTNLV", "RTYRJ", "RDAILY", "HATRIP",
    "MRA", "FRA", "SRA", "GSRA", "NRA", "ISNRA", "ISMRA", "ISFRA",
    "ISHRA", "SPLLVE", "PRPLVE", "PAL", "BAL", "ADVAL", "ACCMAL",
    "RL", "DAILY", "POSDA", "LDAILY",
)
DUPLICATE_CHECK_EXCLUSIONS = ("ATCORR", "LRA", "DA", "CEATWN", "NOK")
DUPLICATE_CEA_FAMILY = (
    "CEA", "CEAHC", "HOSSUB", "HSBTWN", "CHNOK", "CHSRNM", "CHFMNM",
)
DUPLICATE_COLUMN_AB = ("SCCIA", "MEDAL", "UID", "BIOMET")


MODEL_SOURCE_COLUMNS = {
    "id", "status", "unit_code", "fk_unit", "army_no", "fk_employee",
    "check_digit", "do2_group", "do2_type", "do2_no", "do2_item_no",
    "do2_date", "description", "from_date", "to_date", "rate", "tjra",
    "lra", "column_a", "column_b", "column_c", "ref_do2_no",
    "transcription_type", "qe", "created_at", "attachment", "delayed",
    "time_bar_sanction", "mr",
}

def quote_identifier(identifier):
    """Safely quote a PostgreSQL table or column name."""
    return '"' + identifier.replace('"', '""') + '"'


def qualified_table(table_name):
    return (
        f"{quote_identifier(settings.db_schema)}."
        f"{quote_identifier(table_name)}"
    )


def sql_literal_list(values):
    return ", ".join("'" + value.replace("'", "''") + "'" for value in values)


database_inspector = inspect(engine)
select_clause_cache = {}


RULE_SOURCE_REQUIREMENTS = {
    SOURCE_TABLE: {
        "id", "fk_employee", "description", "from_date", "to_date",
        "created_at", "status", "transcription_type",
        "column_a", "column_b", "column_c", "ref_do2_no", "do2_type",
        "do2_no", "do2_item_no", "do2_date", "do2_group",
    },
    "employee": {
        "id", "fs_due_date", "fk_status_code", "status_effect_date",
        "created_at", "current_record", "enrolment_date", "birth_date",
    },
    "status_code": {"id", "status_code"},
    "lpc_register": {"fk_employee", "discharge_date", "created_at"},
    "appendix_j": {
        "id", "description", "do2_group", "column_a", "column_b", "column_c",
        "from_date_req", "to_date_req", "rate_req", "tjra_req", "lra_req",
        "ref_do2_no_req",
    },
    "family_detail": {
        "fk_employee", "relationship", "created_at", "record_status",
    },
}

def table_column_names(table_name):
    return {
        column["name"]
        for column in database_inspector.get_columns(
            table_name,
            schema=settings.db_schema,
        )
    }


def validate_rule_sources():
    """Fail early instead of silently creating incorrect rule features."""
    problems = []
    for table_name, required_columns in RULE_SOURCE_REQUIREMENTS.items():
        available_columns = table_column_names(table_name)
        if not available_columns:
            problems.append(f"missing table: {settings.db_schema}.{table_name}")
            continue
        missing_columns = sorted(required_columns - available_columns)
        if missing_columns:
            problems.append(
                f"{settings.db_schema}.{table_name} missing columns: "
                + ", ".join(missing_columns)
            )
    if problems:
        raise RuntimeError("Rule-feature source validation failed:\n- " + "\n- ".join(problems))


validate_rule_sources()


def make_safe_select_clause(table_name, table_alias="d"):
    """Return aliased columns; dates are text to tolerate extreme old dates."""
    cache_key = (table_name, table_alias)
    if cache_key in select_clause_cache:
        return select_clause_cache[cache_key]

    table_columns = database_inspector.get_columns(
        table_name,
        schema=settings.db_schema,
    )
    if not table_columns:
        raise ValueError(f"Table was not found: {settings.db_schema}.{table_name}")

    expressions = []
    converted_date_columns = []
    alias_sql = quote_identifier(table_alias)
    for column in table_columns:
        column_name = column["name"]
        if table_name == SOURCE_TABLE and column_name not in MODEL_SOURCE_COLUMNS:
            continue
        column_sql = f"{alias_sql}.{quote_identifier(column_name)}"
        output_sql = quote_identifier(column_name)
        if isinstance(column["type"], (Date, DateTime)):
            expressions.append(f"{column_sql}::text AS {output_sql}")
            converted_date_columns.append(column_name)
        else:
            expressions.append(f"{column_sql} AS {output_sql}")

    value = (",\n".join(expressions), converted_date_columns)
    select_clause_cache[cache_key] = value
    return value


def build_enriched_query(table_name, where_sql="", limit_sql=""):
    """
    Build one row per selected DO2 with entry-time-safe N/I/MR/V facts.

    All self-history and one-to-many sources are aggregated before they reach
    pandas. Prior DO2 status and processed_date are never read: do2.status is
    mutable and there is no status-history table. Lower id plus an earlier-or-
    equal created_at is a conservative insertion-order proxy because Java can
    reset created_at during pending reprocessing.
    """
    select_clause, date_columns = make_safe_select_clause(table_name, "d")
    do2_history = qualified_table(SOURCE_TABLE)

    excluded_sql = sql_literal_list(DUPLICATE_CHECK_EXCLUSIONS)
    cea_family_sql = sql_literal_list(DUPLICATE_CEA_FAMILY)
    column_ab_sql = sql_literal_list(DUPLICATE_COLUMN_AB)
    mutex_sql = sql_literal_list(MUTEX_DESCRIPTIONS)

    query = f"""
    WITH selected_do2 AS (
        SELECT source_row.*
        FROM {qualified_table(table_name)} AS source_row
        {where_sql}
        {limit_sql}
    )
    SELECT
        {select_clause},

        CASE
            WHEN employee_row.id IS NOT NULL
             AND employee_row.created_at IS NOT NULL
             AND d.created_at IS NOT NULL
             AND employee_row.created_at <= d.created_at
            THEN 1 ELSE 0
        END AS employee_found,
        CASE WHEN employee_row.created_at <= d.created_at
             THEN employee_row.fs_due_date END::text AS employee_fs_due_date,
        CASE WHEN employee_row.created_at <= d.created_at
                   AND (employee_row.status_effect_date IS NULL
                        OR employee_row.status_effect_date <= d.created_at)
             THEN employee_status.status_code END::text AS employee_status_code,
        CASE WHEN employee_row.created_at <= d.created_at
                   AND employee_row.status_effect_date <= d.created_at
             THEN employee_row.status_effect_date END::text
             AS employee_status_effect_date,
        CASE WHEN employee_row.created_at <= d.created_at
             THEN employee_row.created_at END::text AS employee_created_at,
        CASE WHEN employee_row.created_at <= d.created_at
             THEN employee_row.current_record END::text AS employee_current_record,
        CASE WHEN employee_row.created_at <= d.created_at
             THEN employee_row.enrolment_date END::text AS employee_enrolment_date,
        CASE WHEN employee_row.created_at <= d.created_at
             THEN employee_row.birth_date END::text AS employee_birth_date,

        CASE WHEN d.created_at IS NULL THEN 0 ELSE 1 END
            AS rule_history_cutoff_available,
        COALESCE(lpc_facts.prior_lpc_count, 0) AS rule_prior_lpc_count,
        lpc_facts.latest_prior_discharge_date::text
            AS rule_latest_prior_lpc_discharge_date,

        appendix_j_rule.id AS rule_appendix_j_id,
        appendix_j_rule.column_a::text AS rule_appendix_column_a,
        appendix_j_rule.column_b::text AS rule_appendix_column_b,
        appendix_j_rule.column_c::text AS rule_appendix_column_c,
        appendix_j_rule.from_date_req::text AS rule_appendix_from_date_req,
        appendix_j_rule.to_date_req::text AS rule_appendix_to_date_req,
        appendix_j_rule.rate_req::text AS rule_appendix_rate_req,
        appendix_j_rule.tjra_req::text AS rule_appendix_tjra_req,
        appendix_j_rule.lra_req::text AS rule_appendix_lra_req,
        appendix_j_rule.ref_do2_no_req::text AS rule_appendix_ref_do2_no_req,

        CASE
            WHEN EXISTS (
                SELECT 1
                FROM {qualified_table("appendix_j")} AS appendix_exact
                WHERE UPPER(BTRIM(COALESCE(appendix_exact.description::text, '')))
                      = UPPER(BTRIM(COALESCE(d.description::text, '')))
                  AND (
                      appendix_exact.do2_group IS NULL
                      OR BTRIM(appendix_exact.do2_group::text) = ''
                      OR UPPER(BTRIM(appendix_exact.do2_group::text))
                         = UPPER(BTRIM(COALESCE(d.do2_group::text, '')))
                  )
                  AND UPPER(BTRIM(COALESCE(appendix_exact.column_a::text, '')))
                      = UPPER(BTRIM(COALESCE(d.column_a::text, '')))
                  AND UPPER(BTRIM(COALESCE(appendix_exact.column_b::text, '')))
                      = UPPER(BTRIM(COALESCE(d.column_b::text, '')))
                  AND UPPER(BTRIM(COALESCE(appendix_exact.column_c::text, '')))
                      = UPPER(BTRIM(COALESCE(d.column_c::text, '')))
            ) THEN 1 ELSE 0
        END AS rule_appendix_exact_value_match,

        COALESCE(duplicate_facts.prior_duplicate_candidate_count, 0)
            AS rule_prior_duplicate_candidate_count,
        COALESCE(overlap_facts.prior_mutex_overlap_candidate_count, 0)
            AS rule_prior_mutex_overlap_candidate_count,
        COALESCE(reference_facts.reference_match_count, 0)
            AS rule_reference_match_count,

        COALESCE(pay_history.prior_promotion_candidate_count, 0)
            AS rule_prior_promotion_candidate_count,
        COALESCE(pay_history.prior_optcpc_conflict_candidate_count, 0)
            AS rule_prior_optcpc_conflict_candidate_count,
        COALESCE(pay_history.prior_pay_fix_event_candidate_count, 0)
            AS rule_prior_pay_fix_event_candidate_count,

        COALESCE(family_facts.prior_family_member_count, 0)
            AS rule_prior_family_member_count,
        COALESCE(family_facts.prior_missing_relationship_count, 0)
            AS rule_prior_missing_relationship_count

    FROM selected_do2 AS d
    LEFT JOIN {qualified_table("employee")} AS employee_row
        ON employee_row.id = d.fk_employee
    LEFT JOIN {qualified_table("status_code")} AS employee_status
        ON employee_status.id = employee_row.fk_status_code

    LEFT JOIN LATERAL (
        SELECT
            COUNT(*)::bigint AS prior_lpc_count,
            MAX(lpc.discharge_date) AS latest_prior_discharge_date
        FROM {qualified_table("lpc_register")} AS lpc
        WHERE lpc.fk_employee = d.fk_employee
          AND d.created_at IS NOT NULL
          AND lpc.created_at IS NOT NULL
          AND lpc.created_at < d.created_at
    ) AS lpc_facts ON TRUE

    LEFT JOIN LATERAL (
        SELECT appendix.*
        FROM {qualified_table("appendix_j")} AS appendix
        WHERE UPPER(BTRIM(COALESCE(appendix.description::text, '')))
              = UPPER(BTRIM(COALESCE(d.description::text, '')))
          AND (
              appendix.do2_group IS NULL
              OR BTRIM(appendix.do2_group::text) = ''
              OR UPPER(BTRIM(appendix.do2_group::text))
                 = UPPER(BTRIM(COALESCE(d.do2_group::text, '')))
          )
        ORDER BY
            CASE
                WHEN UPPER(BTRIM(COALESCE(appendix.do2_group::text, '')))
                     = UPPER(BTRIM(COALESCE(d.do2_group::text, '')))
                THEN 0 ELSE 1
            END,
            appendix.id DESC
        LIMIT 1
    ) AS appendix_j_rule ON TRUE

    LEFT JOIN LATERAL (
        SELECT COUNT(history_row.id)::bigint AS prior_duplicate_candidate_count
        FROM {do2_history} AS history_row
        WHERE history_row.fk_employee = d.fk_employee
          AND history_row.id < d.id
          AND d.created_at IS NOT NULL
          AND history_row.created_at IS NOT NULL
          AND history_row.created_at <= d.created_at
          AND UPPER(BTRIM(COALESCE(d.description::text, '')))
              NOT IN ({excluded_sql})
          AND (
              (
                  UPPER(BTRIM(COALESCE(d.description::text, ''))) = 'BUNCH'
                  AND UPPER(BTRIM(COALESCE(history_row.description::text, ''))) = 'BUNCH'
              )
              OR
              (
                  UPPER(BTRIM(COALESCE(d.description::text, ''))) IN ({cea_family_sql})
                  AND history_row.from_date = d.from_date
                  AND UPPER(BTRIM(COALESCE(history_row.description::text, '')))
                      = UPPER(BTRIM(COALESCE(d.description::text, '')))
                  AND history_row.column_a = d.column_a
                  AND history_row.column_b = d.column_b
                  AND history_row.column_c = d.column_c
              )
              OR
              (
                  UPPER(BTRIM(COALESCE(d.description::text, ''))) = 'CANCEL'
                  AND history_row.from_date = d.from_date
                  AND UPPER(BTRIM(COALESCE(history_row.description::text, ''))) = 'CANCEL'
                  AND history_row.column_a = d.column_a
                  AND history_row.ref_do2_no = d.ref_do2_no
              )
              OR
              (
                  UPPER(BTRIM(COALESCE(d.description::text, ''))) IN ({column_ab_sql})
                  AND history_row.from_date = d.from_date
                  AND UPPER(BTRIM(COALESCE(history_row.description::text, '')))
                      = UPPER(BTRIM(COALESCE(d.description::text, '')))
                  AND history_row.column_a = d.column_a
                  AND history_row.column_b = d.column_b
              )
              OR
              (
                  UPPER(BTRIM(COALESCE(d.description::text, '')))
                      NOT IN ('BUNCH', 'CANCEL', {cea_family_sql}, {column_ab_sql})
                  AND history_row.from_date = d.from_date
                  AND UPPER(BTRIM(COALESCE(history_row.description::text, '')))
                      = UPPER(BTRIM(COALESCE(d.description::text, '')))
                  AND history_row.column_a = d.column_a
              )
          )
    ) AS duplicate_facts ON TRUE

    LEFT JOIN LATERAL (
        SELECT COUNT(history_row.id)::bigint AS prior_mutex_overlap_candidate_count
        FROM {do2_history} AS history_row
        WHERE d.from_date IS NOT NULL
          AND UPPER(BTRIM(COALESCE(d.description::text, ''))) IN ({mutex_sql})
          AND history_row.fk_employee = d.fk_employee
          AND history_row.id < d.id
          AND d.created_at IS NOT NULL
          AND history_row.created_at IS NOT NULL
          AND history_row.created_at <= d.created_at
          AND UPPER(BTRIM(COALESCE(history_row.transcription_type::text, ''))) <> 'S'
          AND UPPER(BTRIM(COALESCE(history_row.description::text, ''))) IN ({mutex_sql})
          AND history_row.from_date <= COALESCE(d.to_date, d.from_date)
          AND COALESCE(history_row.to_date, history_row.from_date) >= d.from_date
    ) AS overlap_facts ON TRUE

    LEFT JOIN LATERAL (
        SELECT COUNT(history_row.id)::bigint AS reference_match_count
        FROM {do2_history} AS history_row
        WHERE d.ref_do2_no IS NOT NULL
          AND d.created_at IS NOT NULL
          AND history_row.created_at IS NOT NULL
          AND history_row.created_at <= d.created_at
          AND history_row.id < d.id
          AND history_row.fk_employee = d.fk_employee
          AND BTRIM(d.ref_do2_no::text) = CONCAT(
              BTRIM(history_row.do2_type::text), '/',
              LPAD(BTRIM(history_row.do2_no::text), 4, '0'), '/',
              LPAD(BTRIM(history_row.do2_item_no::text), 3, '0'), '/',
              EXTRACT(YEAR FROM history_row.do2_date)::integer::text
          )
    ) AS reference_facts ON TRUE

    LEFT JOIN LATERAL (
        SELECT
            COUNT(history_row.id) FILTER (
                WHERE UPPER(BTRIM(COALESCE(history_row.description::text, '')))
                      IN ('PROMOT', 'MACPS')
                  AND history_row.from_date = d.from_date
            )::bigint AS prior_promotion_candidate_count,
            COUNT(history_row.id) FILTER (
                WHERE UPPER(BTRIM(COALESCE(history_row.description::text, ''))) = 'OPTCPC'
                  AND UPPER(BTRIM(COALESCE(history_row.column_b::text, ''))) = 'SEVENPC'
                  AND UPPER(BTRIM(COALESCE(history_row.column_a::text, ''))) <> 'INCR'
                  AND history_row.from_date IN (d.from_date, d.to_date)
            )::bigint AS prior_optcpc_conflict_candidate_count,
            COUNT(history_row.id) FILTER (
                WHERE UPPER(BTRIM(COALESCE(history_row.description::text, '')))
                      IN ('OPTFIX', 'PROFIX', 'ACPFIX', 'HAVFIX')
            )::bigint AS prior_pay_fix_event_candidate_count
        FROM {do2_history} AS history_row
        WHERE UPPER(BTRIM(COALESCE(d.description::text, '')))
                  IN ('OPTFIX', 'PROFIX', 'POSIN', 'OPTCPC')
          AND history_row.fk_employee = d.fk_employee
          AND history_row.id < d.id
          AND d.created_at IS NOT NULL
          AND history_row.created_at IS NOT NULL
          AND history_row.created_at <= d.created_at
    ) AS pay_history ON TRUE

    LEFT JOIN LATERAL (
        SELECT
            COUNT(*)::bigint AS prior_family_member_count,
            COUNT(*) FILTER (
                WHERE family_row.relationship IS NULL
                   OR BTRIM(family_row.relationship::text) = ''
            )::bigint AS prior_missing_relationship_count
        FROM {qualified_table("family_detail")} AS family_row
        WHERE family_row.fk_employee = d.fk_employee
          AND d.created_at IS NOT NULL
          AND family_row.created_at IS NOT NULL
          AND family_row.created_at < d.created_at
          AND UPPER(BTRIM(COALESCE(family_row.record_status::text, ''))) NOT IN ('D', 'DELETED')
    ) AS family_facts ON TRUE
    """
    return text(query), date_columns


def sanitize_timestamp_text(series: pd.Series) -> pd.Series:
    """Null out PostgreSQL 'infinity'/'-infinity' sentinel timestamps.

    Columns are cast to text in SQL to tolerate extreme dates, but
    Postgres can also store the literal 'infinity' / '-infinity'
    values, which pandas cannot parse and turns into NaT with no
    explanation. Handling this explicitly here (instead of only in
    the validation step) lets us report and act on it deliberately.
    """
    cleaned = series.astype("string").str.strip().str.lower()
    sentinel_mask = cleaned.isin({"infinity", "-infinity"})
    result = series.copy()
    result.loc[sentinel_mask.fillna(False)] = None
    return result


def parse_datetime_mixed(series: pd.Series) -> pd.Series:
    """Parse mixed PostgreSQL date/timestamp text without format inference loss."""
    sanitized = sanitize_timestamp_text(series)
    try:
        return pd.to_datetime(sanitized, errors="coerce", format="mixed")
    except TypeError:
        # Compatibility fallback for older pandas releases without format="mixed".
        return pd.to_datetime(sanitized, errors="coerce")


def timestamp_quality_masks(series: pd.Series, parsed: pd.Series) -> dict[str, pd.Series]:
    """Separate missing, sentinel, and malformed timestamps for reporting."""
    text_value = series.astype("string").str.strip()
    missing = series.isna() | text_value.eq("").fillna(False)
    sentinel = text_value.str.lower().isin({"infinity", "-infinity"}).fillna(False)
    rejected = parsed.isna()
    malformed = rejected & ~missing & ~sentinel
    return {
        "missing": missing,
        "sentinel": sentinel,
        "malformed": malformed,
        "rejected": rejected,
    }


def normalized_status_sql(alias):
    return (
        "UPPER(BTRIM(REPLACE(COALESCE("
        + alias
        + ".status::text, ''), '\"', '')))"
    )


def load_training_population_prior():
    status_expression = normalized_status_sql("source_row")
    query = text(f"""
        SELECT {status_expression} AS status, COUNT(*)::bigint AS rows
        FROM {qualified_table(SOURCE_TABLE)} AS source_row
        WHERE source_row.{quote_identifier(SPLIT_DATE_COLUMN)} >= CAST(:start_date AS timestamp)
          AND source_row.{quote_identifier(SPLIT_DATE_COLUMN)} < CAST(:end_date AS timestamp)
          AND {status_expression} IN ({MODEL_STATUS_SQL})
          AND UPPER(BTRIM(COALESCE(source_row.do2_group::text, '')))
              IN ({sql_literal_list(VALID_DO2_GROUPS)})
        GROUP BY 1
        ORDER BY 1
    """)
    return pd.read_sql_query(
        query,
        engine,
        params={"start_date": TRAIN_START, "end_date": TRAIN_END},
    )


def load_training_data():
    status_expression = normalized_status_sql("source_row")
    v_status_expression = normalized_status_sql("v_source")
    valid_groups_sql = sql_literal_list(VALID_DO2_GROUPS)

    # All non-V rows are retained. V is sampled only inside the training window.
    where_sql = f"""
        WHERE source_row.{quote_identifier(SPLIT_DATE_COLUMN)} >= CAST(:start_date AS timestamp)
          AND source_row.{quote_identifier(SPLIT_DATE_COLUMN)} < CAST(:end_date AS timestamp)
          AND {status_expression} IN ({LOAD_STATUS_SQL})
          AND UPPER(BTRIM(COALESCE(source_row.do2_group::text, '')))
              IN ({valid_groups_sql})
          AND (
              {status_expression} <> 'V'
              OR source_row.id IN (
                  SELECT sampled_v.id
                  FROM (
                      SELECT v_source.id
                      FROM {qualified_table(SOURCE_TABLE)} AS v_source
                      WHERE v_source.{quote_identifier(SPLIT_DATE_COLUMN)} >= CAST(:start_date AS timestamp)
                        AND v_source.{quote_identifier(SPLIT_DATE_COLUMN)} < CAST(:end_date AS timestamp)
                        AND {v_status_expression} = 'V'
                        AND UPPER(BTRIM(COALESCE(v_source.do2_group::text, '')))
                            IN ({valid_groups_sql})
                      ORDER BY md5(COALESCE(v_source.id::text, ''))
                      LIMIT :max_train_v_rows
                  ) AS sampled_v
              )
          )
    """
    query, date_columns = build_enriched_query(
        SOURCE_TABLE,
        where_sql=where_sql,
    )
    frame = pd.read_sql_query(
        query,
        engine,
        params={
            "start_date": TRAIN_START,
            "end_date": TRAIN_END,
            "max_train_v_rows": int(MAX_TRAIN_V_ROWS),
        },
    )
    print("Training date columns loaded as text:", date_columns)
    print("Training rows enriched with status-independent prior candidate facts.")
    return frame


def load_holdout_data(start_date, end_date, max_rows=None):
    params = {"start_date": start_date, "end_date": end_date}
    status_expression = normalized_status_sql("source_row")
    where_sql = f"""
        WHERE source_row.{quote_identifier(SPLIT_DATE_COLUMN)} >= CAST(:start_date AS timestamp)
          AND source_row.{quote_identifier(SPLIT_DATE_COLUMN)} < CAST(:end_date AS timestamp)
          AND {status_expression} IN ({LOAD_STATUS_SQL})
          AND UPPER(BTRIM(COALESCE(source_row.do2_group::text, '')))
              IN ({sql_literal_list(VALID_DO2_GROUPS)})
    """
    limit_sql = ""
    if max_rows is not None:
        limit_sql = """
            ORDER BY md5(COALESCE(source_row.id::text, ''))
            LIMIT :max_rows
        """
        params["max_rows"] = int(max_rows)

    query, date_columns = build_enriched_query(
        SOURCE_TABLE,
        where_sql=where_sql,
        limit_sql=limit_sql,
    )
    frame = pd.read_sql_query(query, engine, params=params)
    print("Holdout date columns loaded as text:", date_columns)
    print("Holdout rows enriched with status-independent prior candidate facts.")
    return frame


def enriched_cache_path(split_name, start_date, end_date, row_cap):
    cap_text = "all" if row_cap is None else str(int(row_cap))
    filename = (
        f"{DATA_CACHE_VERSION}_{split_name}_"
        f"{start_date}_{end_date}_{cap_text}.parquet"
    ).replace(":", "-")
    return DATA_CACHE_DIR / filename


def load_or_cache_enriched(
    split_name,
    start_date,
    end_date,
    row_cap,
    loader,
):
    cache_path = enriched_cache_path(
        split_name, start_date, end_date, row_cap
    )
    if USE_DATA_CACHE and cache_path.exists() and not REFRESH_DATA_CACHE:
        print(f"Loading cached {split_name} data:", cache_path)
        started = time.perf_counter()
        frame = pd.read_parquet(cache_path)
        print(
            f"Loaded cached {split_name} rows in "
            f"{time.perf_counter() - started:,.1f} seconds"
        )
        return frame

    started = time.perf_counter()
    frame = loader()
    print(
        f"Database enrichment for {split_name} finished in "
        f"{(time.perf_counter() - started) / 60:,.1f} minutes"
    )
    if USE_DATA_CACHE:
        DATA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        frame.to_parquet(cache_path, index=False)
        print(f"Cached enriched {split_name} data:", cache_path)
    return frame


def show_recommended_index_sql():
    print("Recommended one-time PostgreSQL indexes for faster enrichment:")
    print(f"""
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_do2_ml_employee_created_id
    ON {qualified_table(SOURCE_TABLE)} (fk_employee, created_at, id);
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_do2_ml_split_selection
    ON {qualified_table(SOURCE_TABLE)} (created_at, do2_group, id);
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_do2_ml_employee_desc_from_created
    ON {qualified_table(SOURCE_TABLE)} (fk_employee, description, from_date, created_at);
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_lpc_ml_employee_created
    ON {qualified_table('lpc_register')} (fk_employee, created_at);
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_family_ml_employee_created
    ON {qualified_table('family_detail')} (fk_employee, created_at);
""".strip())


training_population_prior = load_training_population_prior()
show_recommended_index_sql()
train_raw = load_or_cache_enriched(
    "train",
    TRAIN_START,
    TRAIN_END,
    MAX_TRAIN_V_ROWS,
    load_training_data,
)
validation_raw = load_or_cache_enriched(
    "validation",
    VALIDATION_START,
    VALIDATION_END,
    MAX_VALIDATION_ROWS,
    lambda: load_holdout_data(
        start_date=VALIDATION_START,
        end_date=VALIDATION_END,
        max_rows=MAX_VALIDATION_ROWS,
    ),
)

print("Raw training shape:", train_raw.shape)
print("Raw validation shape:", validation_raw.shape)
display(training_population_prior)


Recommended one-time PostgreSQL indexes for faster enrichment:
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_do2_ml_employee_created_id
    ON "public"."do2" (fk_employee, created_at, id);
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_do2_ml_split_selection
    ON "public"."do2" (created_at, do2_group, id);
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_do2_ml_employee_desc_from_created
    ON "public"."do2" (fk_employee, description, from_date, created_at);
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_lpc_ml_employee_created
    ON "public"."lpc_register" (fk_employee, created_at);
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_family_ml_employee_created
    ON "public"."family_detail" (fk_employee, created_at);
Loading cached train data: /home/ruthikreddy/Desktop/do2/do2_status_prediction/data/model_cache/time_safe_v6_status_independent_history_train_2022-01-01_2025-01-01_250000.parquet
Loaded cached train rows in 0.8 seconds
Loading cached validation data: /home/ruthikreddy/Desktop/do2/do2_sta

,status,rows
0,I,82801
1,MR,32380
2,N,7084
3,V,2110091


## 4. Clean labels without target-dependent holdout filtering

In [4]:
def normalize_status(series: pd.Series) -> pd.Series:
    """Standardize status values before filtering."""
    return (
        series.fillna("")
        .astype("string")
        .str.replace('"', "", regex=False)
        .str.strip()
        .str.upper()
    )


REQUIRED_V_COLUMNS = (
    "unit_code", "army_no", "fk_employee", "do2_group", "do2_type",
    "do2_no", "do2_item_no", "do2_date", "created_at", "description",
    "transcription_type", "qe", "check_digit",
)


def missing_value_mask(series):
    missing = series.isna()
    if series.dtype == "object" or str(series.dtype).startswith("string"):
        missing = missing | series.fillna("").astype(str).str.strip().eq("")
    return missing


def clean_split(frame, split_name, *, drop_incomplete_v_labels=False):
    cleaned = frame.copy()
    initial_rows = len(cleaned)
    cleaned["status"] = normalize_status(cleaned["status"])

    # Rows whose created_at cannot be parsed (NULL, malformed, or a
    # Postgres infinity/-infinity sentinel) cannot be placed in a
    # time-safe split. Remove them here, explicitly, with a count in the
    # report — instead of letting them silently become NaT and blow up
    # the split_validate check downstream.
    created_at_original = cleaned[SPLIT_DATE_COLUMN].copy()
    created_at_parsed = parse_datetime_mixed(created_at_original)
    timestamp_masks = timestamp_quality_masks(
        created_at_original,
        created_at_parsed,
    )
    unparseable_created_at_rows = int(timestamp_masks["rejected"].sum())
    missing_created_at_rows = int(timestamp_masks["missing"].sum())
    sentinel_created_at_rows = int(timestamp_masks["sentinel"].sum())
    malformed_created_at_rows = int(timestamp_masks["malformed"].sum())

    if unparseable_created_at_rows:
        rejected_examples = (
            created_at_original.loc[timestamp_masks["rejected"]]
            .astype("string")
            .fillna("<NULL>")
            .value_counts(dropna=False)
            .head(10)
            .rename_axis("rejected_created_at")
            .reset_index(name="rows")
        )
        print(
            f"{split_name}: rejected {unparseable_created_at_rows:,} created_at "
            "values after mixed-format parsing; top examples:"
        )
        display(rejected_examples)

    if DROP_UNPARSEABLE_CREATED_AT_ROWS:
        cleaned = cleaned.loc[created_at_parsed.notna()].copy()
    cleaned[SPLIT_DATE_COLUMN] = created_at_parsed.loc[cleaned.index]

    valid_status_mask = cleaned["status"].isin(VALID_STATUSES)
    excluded_status_rows = int((~valid_status_mask).sum())

    excluded_lifecycle_status_rows = int(

        cleaned["status"].isin(set(LOAD_STATUSES) - set(VALID_STATUSES)).sum()

    )

    cleaned = cleaned.loc[valid_status_mask].copy()


    normalized_group = (
        cleaned["do2_group"].fillna("").astype("string").str.strip().str.upper()
    )
    valid_group_mask = normalized_group.isin(VALID_DO2_GROUPS)
    excluded_do2_group_rows = int((~valid_group_mask).sum())
    cleaned = cleaned.loc[valid_group_mask].copy()

    missing_required = pd.DataFrame(
        {
            column: missing_value_mask(cleaned[column])
            for column in REQUIRED_V_COLUMNS
            if column in cleaned.columns
        },
        index=cleaned.index,
    ).any(axis=1)

    incomplete_v_rows_found = int(
        (cleaned["status"].eq("V") & missing_required).sum()
    )
    incomplete_v_rows_removed = 0
    if drop_incomplete_v_labels:
        remove_mask = cleaned["status"].eq("V") & missing_required
        incomplete_v_rows_removed = int(remove_mask.sum())
        cleaned = cleaned.loc[~remove_mask].copy()

    cleaned = cleaned.reset_index(drop=True)
    report = {
        "split": split_name,
        "initial_rows": initial_rows,
        "unparseable_created_at_rows": unparseable_created_at_rows,
        "missing_created_at_rows": missing_created_at_rows,
        "sentinel_created_at_rows": sentinel_created_at_rows,
        "malformed_created_at_rows": malformed_created_at_rows,
        "excluded_status_rows": excluded_status_rows,
        "excluded_lifecycle_status_rows": excluded_lifecycle_status_rows,
        "excluded_do2_group_rows": excluded_do2_group_rows,
        "incomplete_v_rows_found": incomplete_v_rows_found,
        "incomplete_v_rows_removed": incomplete_v_rows_removed,
        "final_rows": len(cleaned),
    }
    return cleaned, report


train_df, train_cleaning_report = clean_split(
    train_raw,
    "train",
    drop_incomplete_v_labels=DROP_INCOMPLETE_V_TRAINING_LABELS,
)
validation_df, validation_cleaning_report = clean_split(
    validation_raw,
    "validation",
    drop_incomplete_v_labels=False,
)

cleaning_report = pd.DataFrame(
    [train_cleaning_report, validation_cleaning_report]
)
display(cleaning_report)

del train_raw, validation_raw
_ = gc.collect()


,split,initial_rows,unparseable_created_at_rows,missing_created_at_rows,sentinel_created_at_rows,malformed_created_at_rows,excluded_status_rows,excluded_lifecycle_status_rows,excluded_do2_group_rows,incomplete_v_rows_found,incomplete_v_rows_removed,final_rows
0,train,597078,0,0,0,0,224813,224813,0,0,0,372265
1,validation,250000,0,0,0,0,12255,12255,0,1149,0,237745


## 5. Validate split grain, dates, classes, and overlap

In [5]:
def split_profile(frame, split_name):
    return (
        frame["status"]
        .value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="rows")
        .assign(
            split=split_name,
            percentage=lambda value: value["rows"] / value["rows"].sum() * 100,
        )[["split", "status", "rows", "percentage"]]
    )


def validate_split(frame, split_name, start_date, end_date):
    if "id" not in frame:
        raise ValueError(f"{split_name} is missing id")
    duplicate_ids = int(frame["id"].duplicated().sum())
    if duplicate_ids:
        raise ValueError(
            f"{split_name} contains {duplicate_ids:,} duplicated ids after joins"
        )

    timestamps = pd.to_datetime(frame[SPLIT_DATE_COLUMN], errors="coerce")
    missing_timestamps = int(timestamps.isna().sum())
    outside_window = int(
        (~timestamps.between(
            pd.Timestamp(start_date),
            pd.Timestamp(end_date),
            inclusive="left",
        )).sum()
    )
    if missing_timestamps or outside_window:
        raise ValueError(
            f"{split_name} timestamp validation failed: "
            f"missing={missing_timestamps:,}, outside_window={outside_window:,}"
        )
    return {
        "split": split_name,
        "rows": int(len(frame)),
        "unique_ids": int(frame["id"].nunique()),
        "minimum_created_at": timestamps.min(),
        "maximum_created_at": timestamps.max(),
    }


split_checks = pd.DataFrame([
    validate_split(train_df, "train", TRAIN_START, TRAIN_END),
    validate_split(
        validation_df, "validation", VALIDATION_START, VALIDATION_END
    ),
])

TRAIN_IDS = frozenset(train_df["id"].tolist())
VALIDATION_IDS = frozenset(validation_df["id"].tolist())
train_validation_overlap = len(TRAIN_IDS & VALIDATION_IDS)
if train_validation_overlap:
    raise ValueError(
        f"Train/validation id overlap: {train_validation_overlap:,} rows"
    )

status_profile = pd.concat(
    [
        split_profile(train_df, "train"),
        split_profile(validation_df, "validation"),
    ],
    ignore_index=True,
)
display(split_checks)
display(status_profile)
print("Train/validation id overlap:", train_validation_overlap)


,split,rows,unique_ids,minimum_created_at,maximum_created_at
0,train,372265,372265,2022-01-03 17:04:15.160000,2024-12-31 16:25:23.533
1,validation,237745,237745,2025-01-01 10:44:56.681978,2025-06-30 18:24:29.335


,split,status,rows,percentage
0,train,V,250000,67.156461
1,train,I,82801,22.242489
2,train,MR,32380,8.698105
3,train,N,7084,1.902945
4,validation,V,187687,78.944668
5,validation,I,48305,20.317988
6,validation,MR,1122,0.471934
7,validation,N,631,0.26541


Train/validation id overlap: 0


## 6. Build one reproducible feature set

In [6]:

from dataclasses import dataclass

import numpy as np
import pandas as pd


FEATURE_SET_VERSION = "evidence_safe_rule_features_v6"
MISSING_CATEGORY = "__MISSING__"


LEAKAGE_COLUMNS = frozenset(
    {
        "status", "target", "is_non_valid", "reason", "sys_reason",
        "change_reason", "rejection_type", "approved", "reviewed",
        "accepted_qe", "review_qe", "fk_auditor", "fk_aao", "fk_ao",
        "fk_hoo", "auditor_date", "aao_date", "ao_date", "hoo_date",
        "processed_date", "bulk_approval", "attachment_verified",
        "udise_aud_verified", "udise_aao_verified", "udise_sao_verified",
        "fk_remove_mr_ao", "fk_remove_mr_ao_date", "mr_removal_reason",
        "fk_put_mr_ao", "fk_put_mr_ao_date", "mr_put_reason", "fk_task",
        "sms_sent_at", "pan", "sms_sent", "fk_usr", "remarks", "audit_cage",
    }
)


# Only columns needed for cleaning, verified rules, or point-in-time joins.
CANDIDATE_COLUMNS = [
    "unit_code", "fk_unit", "army_no", "fk_employee", "check_digit",
    "do2_group", "do2_type", "do2_no", "do2_item_no", "do2_date",
    "description", "from_date", "to_date", "rate", "tjra", "lra",
    "column_a", "column_b", "column_c", "ref_do2_no",
    "transcription_type", "qe", "created_at", "attachment", "delayed",
    "time_bar_sanction", "mr",
]


# Retained for V-label cleaning and source joins even when not direct features.
REQUIRED_INPUT_COLUMNS = [
    "unit_code", "army_no", "fk_employee", "do2_group", "do2_type",
    "do2_no", "do2_item_no", "do2_date", "created_at", "description",
    "transcription_type", "qe", "check_digit",
]


# Stable business categories only; identifiers and QE are not direct features.
CATEGORICAL_BASE = [
    "do2_group", "do2_type", "description", "transcription_type",
    "delayed", "time_bar_sanction",
]
NUMERIC_BASE = ["rate", "tjra", "lra"]
TEXT_BASE: list[str] = []

FEATURE_CONFIG = {
    "candidate_columns": CANDIDATE_COLUMNS,
    "required_input_columns": REQUIRED_INPUT_COLUMNS,
    "categorical_base": CATEGORICAL_BASE,
    "numeric_base": NUMERIC_BASE,
    "text_base": TEXT_BASE,
}

# Backward-compatible audit list used by the notebook metadata/report cells.
# These names are no longer generated, rather than generated and then dropped.
FEATURES_TO_DROP = frozenset(
    {
        "do2_item_no",
        "unit_code",
        "check_digit",
        "punishment",
        "qe",
        "do2_group_type",
        "unit_do2_type",
        "description_do2_type",
        "description_char_count",
        "description_word_count",
        "description_has_digit",
        "rate_is_negative",
        "tjra_is_negative",
        "lra_is_negative",
        "rate_log_abs",
        "tjra_log_abs",
        "lra_log_abs",
        "rate_tjra_abs_diff",
        "rate_lra_abs_diff",
        "do2_date_plausible",
        "from_date_plausible",
        "to_date_plausible",
        "created_at_plausible",
        "army_no_is_placeholder",
        "abs_duration_days",
        "log_abs_duration_days",
        "abs_submission_delay_days",
        "log_abs_submission_delay_days",
        "created_before_do2",
        "same_day_created_do2",
        "zero_duration",
        "long_duration",
        "core_required_present_count",
        "conditional_required_present_count",
        "conditional_required_missing_count",
        "has_punishment",
        "has_pto_remarks",
        "has_column9_remarks",
        "has_cfa_sanction_received",
        "has_cfa_sanction_letter_no",
        "has_cfa_sanction_date",
        "prior_batch_row_count",
        "prior_do2_no_row_count",
        "prior_army_do2_type_row_count",
        "prior_unit_do2_type_row_count",
        "prior_batch_do2_item_share",
    }
)

CORE_REQUIRED_COLUMNS = (
    "army_no", "check_digit", "fk_employee", "unit_code", "fk_unit",
    "do2_group", "do2_type", "do2_no", "do2_item_no", "do2_date",
    "description", "from_date", "transcription_type",
)

APPLICATION_EFFECTIVE_STATUS_CODES = frozenset({"E", "N", "R", "D", "M", "W", "T"})
ABOLISHED_DESCRIPTION_SET = frozenset(
    {
        "CILQ", "SNLQ", "FAA", "STRINC", "HCA", "FLCHCA", "ADALLC",
        "ACTGA", "COMNDO", "INSTAL", "OUTFIT", "HSINC", "HCWA",
        "CONDMT", "BOUNTY", "RLCA1", "RLCA2", "RLCA3", "RLCA4",
        "TYOJ", "TYRJ", "DAILY",
    }
)


# Documentation only: deliberately excluded weak/redundant groups.
REMOVED_FEATURE_GROUPS = (
    "identifier-derived and in-memory history features",
    "description character/word/digit counts",
    "raw categorical pair interactions",
    "log/absolute amount transforms and amount-difference proxies",
    "absolute/log duration transforms",
    "QE/input-QE parsing and quarter-distance features",
    "global conditional-field missing counts",
    "free-text remark presence flags",
    "unused CFA sanction and punishment fields",
    "duplicate rule counts when only existence is needed",
    "unscoped service-age thresholds",
)


@dataclass
class FeatureResult:
    frame: pd.DataFrame
    categorical_columns: list[str]
    text_columns: list[str]


def normalize_status(series: pd.Series) -> pd.Series:
    return (
        series.fillna("")
        .astype("string")
        .str.replace('"', "", regex=False)
        .str.strip()
        .str.upper()
    )


def supervised_status_mask(frame: pd.DataFrame) -> pd.Series:
    if "status" not in frame:
        raise KeyError("status is required for supervised training/evaluation")
    return normalize_status(frame["status"]).ne("")


def filter_supervised_rows(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.loc[supervised_status_mask(frame)].reset_index(drop=True)


def leakage_columns_present(columns: list[str] | pd.Index) -> list[str]:
    present = set(columns)
    return sorted(column for column in LEAKAGE_COLUMNS if column in present)


def validate_prediction_columns(frame: pd.DataFrame, required_columns: list[str]) -> None:
    missing = sorted(set(required_columns) - set(frame.columns))
    if missing:
        raise ValueError(
            "Prediction input is missing required entry-time columns: "
            + ", ".join(missing)
        )


def _series(frame: pd.DataFrame, column: str, default=np.nan) -> pd.Series:
    if column in frame:
        return frame[column]
    return pd.Series(default, index=frame.index)


def _clean_category(series: pd.Series) -> pd.Series:
    cleaned = series.astype("string").str.strip().str.upper()
    return cleaned.fillna(MISSING_CATEGORY).replace("", MISSING_CATEGORY).astype(str)


def _normalized_text(frame: pd.DataFrame, column: str) -> pd.Series:
    return (
        _series(frame, column, "")
        .fillna("")
        .astype("string")
        .str.strip()
        .str.upper()
    )


def _present(series: pd.Series) -> pd.Series:
    return (
        series.notna()
        & series.astype("string").str.strip().fillna("").ne("")
    ).astype("int8")


def _column_present(frame: pd.DataFrame, column: str) -> pd.Series:
    return _present(_series(frame, column))


def _numeric_source(frame: pd.DataFrame, column: str) -> pd.Series:
    return pd.to_numeric(_series(frame, column), errors="coerce").astype("float32")


def _truthy_source(frame: pd.DataFrame, column: str) -> pd.Series:
    return _normalized_text(frame, column).isin(
        {"1", "TRUE", "T", "YES", "Y", "REQUIRED", "R"}
    )


def _date(frame: pd.DataFrame, column: str) -> pd.Series:
    return parse_datetime_mixed(_series(frame, column))


def _rule_bool(condition: pd.Series, available: pd.Series | None = None) -> pd.Series:
    output = condition.astype("float32")
    if available is not None:
        output = output.where(available, np.nan)
    return output.astype("float32")


def build_status_n_features(frame):
    features = pd.DataFrame(index=frame.index)

    from_date = _date(frame, "from_date")
    to_date = _date(frame, "to_date")
    created_at = _date(frame, "created_at")
    employee_fs_due_date = _date(frame, "employee_fs_due_date")
    lpc_discharge_date = _date(frame, "rule_latest_prior_lpc_discharge_date")
    status_effect_date = _date(frame, "employee_status_effect_date")
    employee_created_at = _date(frame, "employee_created_at")

    employee_found = _numeric_source(frame, "employee_found").fillna(0).gt(0)
    status_code = (
        frame.get("employee_status_code", pd.Series("", index=frame.index))
        .fillna("")
        .astype("string")
        .str.strip()
        .str.upper()
    )
    status_available = employee_found & status_code.ne("")

    # The processor uses an LPC discharge date as an FS-date fallback.
    effective_fs_due_date = employee_fs_due_date.fillna(lpc_discharge_date)
    fs_date_available = from_date.notna() & effective_fs_due_date.notna()
    to_fs_date_available = to_date.notna() & effective_fs_due_date.notna()

    features["n_employee_found"] = employee_found.astype("int8")
    features["n_employee_status_missing"] = (~status_available).astype("int8")
    features["n_employee_effective_by_application_rule"] = _rule_bool(
        status_code.isin(APPLICATION_EFFECTIVE_STATUS_CODES),
        status_available,
    )
    features["n_employee_not_effective_by_application_rule"] = _rule_bool(
        ~status_code.isin(APPLICATION_EFFECTIVE_STATUS_CODES),
        status_available,
    )

    features["n_fs_due_date_missing"] = employee_fs_due_date.isna().astype("int8")
    features["n_fs_due_date_from_prior_lpc"] = (
        employee_fs_due_date.isna() & lpc_discharge_date.notna()
    ).astype("int8")
    features["n_effective_fs_due_date_missing"] = effective_fs_due_date.isna().astype("int8")
    features["n_days_from_fs_due_date"] = (
        from_date.dt.normalize() - effective_fs_due_date.dt.normalize()
    ).dt.days.astype("float32")
    features["n_days_after_fs_due_date"] = (
        features["n_days_from_fs_due_date"].clip(lower=0).astype("float32")
    )
    features["n_from_date_after_fs_due_date"] = _rule_bool(
        from_date.gt(effective_fs_due_date),
        fs_date_available,
    )
    features["n_from_date_within_fs_month"] = _rule_bool(
        from_date.dt.to_period("M").eq(effective_fs_due_date.dt.to_period("M")),
        fs_date_available,
    )
    features["n_to_date_after_fs_due_date"] = _rule_bool(
        to_date.gt(effective_fs_due_date),
        to_fs_date_available,
    )

    description = _clean_category(
        frame.get("description", pd.Series("", index=frame.index))
    )
    features["n_description_is_death"] = description.eq("DEATH").astype("int8")
    features["n_description_is_atcorr"] = description.eq("ATCORR").astype("int8")
    features["n_description_is_optfix"] = description.eq("OPTFIX").astype("int8")
    features["n_description_is_cea"] = description.isin({"CEA", "CEAHC"}).astype("int8")
    features["n_prior_lpc_count"] = _numeric_source(frame, "rule_prior_lpc_count").fillna(0)
    features["n_prior_lpc_exists"] = features["n_prior_lpc_count"].gt(0).astype("int8")

    return features


MUTEX_DESCRIPTION_SET = frozenset({
    "TYOJ", "TYRJ", "DRDSRA", "AL", "CL", "SL", "EXTNAL", "EXTNCL",
    "EXTNSL", "TJRA", "RTYOJ", "EXTNLV", "RTYRJ", "RDAILY", "HATRIP",
    "MRA", "FRA", "SRA", "GSRA", "NRA", "ISNRA", "ISMRA", "ISFRA",
    "ISHRA", "SPLLVE", "PRPLVE", "PAL", "BAL", "ADVAL", "ACCMAL",
    "RL", "DAILY", "POSDA", "LDAILY",
})
DUPLICATE_CEA_FAMILY_SET = frozenset({
    "CEA", "CEAHC", "HOSSUB", "HSBTWN", "CHNOK", "CHSRNM", "CHFMNM",
})
DUPLICATE_COLUMN_AB_SET = frozenset({"SCCIA", "MEDAL", "UID", "BIOMET"})
DUPLICATE_EXCLUDED_SET = frozenset({"ATCORR", "LRA", "DA", "CEATWN", "NOK"})


def build_status_i_features(frame):
    features = pd.DataFrame(index=frame.index)
    description = _clean_category(
        frame.get("description", pd.Series("", index=frame.index))
    )
    from_date = _date(frame, "from_date")
    employee_fs_due_date = _date(frame, "employee_fs_due_date")
    enrolment_date = _date(frame, "employee_enrolment_date")
    birth_date = _date(frame, "employee_birth_date")

    features["i_description_abolished_on_from_date"] = (
        description.isin(ABOLISHED_DESCRIPTION_SET)
        & from_date.ge(pd.Timestamp("2017-07-01"))
    ).astype("int8")
    features["i_days_after_7cpc_abolition"] = (
        from_date - pd.Timestamp("2017-07-01")
    ).dt.days.where(description.isin(ABOLISHED_DESCRIPTION_SET)).astype("float32")

    duplicate_group = pd.Series("DEFAULT", index=frame.index, dtype="object")
    duplicate_group.loc[description.isin(DUPLICATE_EXCLUDED_SET)] = "EXCLUDED"
    duplicate_group.loc[description.eq("BUNCH")] = "BUNCH"
    duplicate_group.loc[description.isin(DUPLICATE_CEA_FAMILY_SET)] = "CEA_FAMILY"
    duplicate_group.loc[description.eq("CANCEL")] = "CANCEL"
    duplicate_group.loc[description.isin(DUPLICATE_COLUMN_AB_SET)] = "COLUMN_AB"
    features["i_duplicate_rule_group"] = duplicate_group
    features["i_duplicate_check_applicable"] = duplicate_group.ne("EXCLUDED").astype("int8")

    count_sources = {
        "i_prior_duplicate_candidate_count": "rule_prior_duplicate_candidate_count",
        "i_prior_mutex_overlap_candidate_count": "rule_prior_mutex_overlap_candidate_count",
        "i_reference_match_count": "rule_reference_match_count",
        "i_prior_promotion_candidate_count": "rule_prior_promotion_candidate_count",
        "i_prior_optcpc_conflict_candidate_count": "rule_prior_optcpc_conflict_candidate_count",
        "i_prior_pay_fix_event_candidate_count": "rule_prior_pay_fix_event_candidate_count",
        "i_prior_family_member_count": "rule_prior_family_member_count",
        "i_prior_missing_relationship_count": "rule_prior_missing_relationship_count",
    }
    for feature_name, source_name in count_sources.items():
        features[feature_name] = _numeric_source(frame, source_name).fillna(0)

    features["i_has_prior_duplicate_candidate"] = features["i_prior_duplicate_candidate_count"].gt(0).astype("int8")
    features["i_description_requires_mutex_check"] = description.isin(
        MUTEX_DESCRIPTION_SET
    ).astype("int8")
    features["i_has_prior_mutex_overlap_candidate"] = (
        features["i_prior_mutex_overlap_candidate_count"].gt(0).astype("int8")
    )

    reference_text = (
        frame.get("ref_do2_no", pd.Series("", index=frame.index))
        .fillna("")
        .astype("string")
        .str.strip()
    )
    features["i_reference_present"] = reference_text.ne("").astype("int8")
    features["i_reference_format_valid"] = (
        (
            reference_text.str.fullmatch(r"[^/]+/\d{4}/\d{3}/\d{4}", na=False)
        ).astype("int8")
    )
    features["i_reference_found"] = features["i_reference_match_count"].gt(0).astype("int8")

    fs_available = from_date.notna() & employee_fs_due_date.notna()
    features["i_from_date_within_fs_month"] = _rule_bool(
        from_date.dt.to_period("M").eq(employee_fs_due_date.dt.to_period("M")),
        fs_available,
    )
    features["i_is_optfix_or_profix"] = description.isin(
        {"OPTFIX", "PROFIX"}
    ).astype("int8")
    features["i_pay_fixation_candidate_support_missing"] = (
        features["i_is_optfix_or_profix"].eq(1)
        & features["i_prior_promotion_candidate_count"].eq(0)
    ).astype("int8")
    features["i_has_prior_optcpc_conflict_candidate"] = (
        features["i_prior_optcpc_conflict_candidate_count"].gt(0).astype("int8")
    )

    # Appendix-J required-field checks use only entry-time DO2 values.
    appendix_rule_found = (
        _numeric_source(frame, "rule_appendix_j_id").notna()
        & USE_CURRENT_APPENDIX_J_FOR_HISTORICAL_ROWS
    )
    required_checks = {
        "from_date": "rule_appendix_from_date_req",
        "to_date": "rule_appendix_to_date_req",
        "rate": "rule_appendix_rate_req",
        "tjra": "rule_appendix_tjra_req",
        "lra": "rule_appendix_lra_req",
        "ref_do2_no": "rule_appendix_ref_do2_no_req",
    }
    appendix_failures = pd.DataFrame(index=frame.index)
    for source_column, rule_column in required_checks.items():
        required = (
            _truthy_source(frame, rule_column)
            & USE_CURRENT_APPENDIX_J_FOR_HISTORICAL_ROWS
        )
        missing = _column_present(frame, source_column).eq(0)
        appendix_failures[source_column] = required & missing
        features[f"i_appendix_required_{source_column}_missing"] = (
            required & missing
        ).astype("int8")
    features["i_appendix_j_rule_found"] = appendix_rule_found.astype("int8")
    features["i_appendix_required_missing_count"] = (
        appendix_failures.sum(axis=1).astype("float32")
    )
    features["i_relationship_value_missing"] = _column_present(
        frame, "column_b"
    ).eq(0).astype("int8")

    features["i_service_days_at_from_date"] = (
        from_date - enrolment_date
    ).dt.days.astype("float32")
    features["i_age_days_at_from_date"] = (
        from_date - birth_date
    ).dt.days.astype("float32")
    features["i_less_than_six_months_service"] = _rule_bool(
        features["i_service_days_at_from_date"].lt(183),
        features["i_service_days_at_from_date"].notna(),
    )
    features["i_less_than_eight_years_service"] = _rule_bool(
        features["i_service_days_at_from_date"].lt(8 * 365.25),
        features["i_service_days_at_from_date"].notna(),
    )

    features["i_rule_history_cutoff_missing"] = _numeric_source(
        frame, "rule_history_cutoff_available"
    ).fillna(0).eq(0).astype("int8")
    return features


def _normalized_rule_text(frame, column):
    if column not in frame:
        return pd.Series("", index=frame.index, dtype="string")
    return (
        frame[column]
        .fillna("")
        .astype("string")
        .str.strip()
        .str.upper()
    )


def build_status_mr_features(frame):
    """Manual-rejection facts confirmed by Java/application evidence."""
    features = pd.DataFrame(index=frame.index)
    appendix_rule_found = (
        _numeric_source(frame, "rule_appendix_j_id").notna()
        & USE_CURRENT_APPENDIX_J_FOR_HISTORICAL_ROWS
    )
    appendix_exact_match = _numeric_source(
        frame, "rule_appendix_exact_value_match"
    ).fillna(0).gt(0)
    features["mr_entry_flag"] = _truthy_source(frame, "mr").astype("int8")
    features["mr_appendix_value_combination_invalid"] = (
        appendix_rule_found & ~appendix_exact_match
    ).astype("int8")
    return features


def build_status_v_features(frame):
    """Verified static acceptance evidence; no outcome or workflow columns."""
    features = pd.DataFrame(index=frame.index)
    description = _clean_category(
        frame.get("description", pd.Series("", index=frame.index))
    )
    from_date = _date(frame, "from_date")
    features["v_appt_nonactionable_after_7cpc"] = (
        description.eq("APPT") & from_date.ge(pd.Timestamp("2017-07-01"))
    ).astype("int8")

    appendix_rule_found = (
        _numeric_source(frame, "rule_appendix_j_id").notna()
        & USE_CURRENT_APPENDIX_J_FOR_HISTORICAL_ROWS
    )
    appendix_exact_match = _numeric_source(
        frame, "rule_appendix_exact_value_match"
    ).fillna(0).gt(0)
    static_failures = pd.DataFrame(index=frame.index)
    static_failures["abolished"] = (
        description.isin(ABOLISHED_DESCRIPTION_SET)
        & from_date.ge(pd.Timestamp("2017-07-01"))
    )
    static_failures["negative_duration"] = (
        _date(frame, "from_date").notna()
        & _date(frame, "to_date").notna()
        & _date(frame, "from_date").gt(_date(frame, "to_date"))
    )
    static_failures["appendix_combination"] = appendix_rule_found & ~appendix_exact_match
    features["v_verified_static_violation_count"] = (
        static_failures.sum(axis=1).astype("float32")
    )
    features["v_all_verified_static_checks_passed"] = (
        features["v_verified_static_violation_count"].eq(0).astype("int8")
    )
    return features


def build_features(
    frame: pd.DataFrame,
    *,
    expected_columns: list[str] | None = None,
    expected_categorical: list[str] | None = None,
    expected_text: list[str] | None = None,
) -> FeatureResult:
    """Build the same compact feature matrix for every data split."""
    del expected_text  # Native CatBoost text handling is intentionally disabled.
    result = pd.DataFrame(index=frame.index)
    categorical: list[str] = []

    configured = set(CATEGORICAL_BASE) | set(NUMERIC_BASE)
    leaking_features = sorted(configured & set(leakage_columns_present(frame.columns)))
    if leaking_features:
        raise RuntimeError(f"Leakage columns configured as model features: {leaking_features}")

    for column in CATEGORICAL_BASE:
        if column in frame:
            result[column] = _clean_category(frame[column])
            categorical.append(column)

    for column in NUMERIC_BASE:
        value = pd.to_numeric(_series(frame, column), errors="coerce")
        result[column] = value.astype("float32")
        result[f"{column}_is_missing"] = value.isna().astype("int8")
        result[f"{column}_is_zero"] = value.eq(0).fillna(False).astype("int8")

    do2_date = _date(frame, "do2_date")
    from_date = _date(frame, "from_date")
    to_date = _date(frame, "to_date")
    created_at = _date(frame, "created_at")

    result["duration_days"] = (to_date - from_date).dt.days.astype("float32")
    result["submission_delay_days"] = (
        created_at.dt.normalize() - do2_date.dt.normalize()
    ).dt.days.astype("float32")
    result["negative_duration"] = (
        from_date.notna() & to_date.notna() & from_date.gt(to_date)
    ).astype("int8")
    result["from_date_after_do2_date"] = (
        from_date.notna() & do2_date.notna()
        & from_date.dt.normalize().gt(do2_date.dt.normalize())
    ).astype("int8")

    for feature_name, source_column in {
        "has_reference_do2": "ref_do2_no",
        "has_attachment": "attachment",
        "has_column_a": "column_a",
        "has_column_b": "column_b",
        "has_column_c": "column_c",
    }.items():
        result[feature_name] = _column_present(frame, source_column)

    core_presence = pd.DataFrame(
        {column: _column_present(frame, column) for column in CORE_REQUIRED_COLUMNS},
        index=frame.index,
    )
    result["missing_required_field_count"] = (
        len(CORE_REQUIRED_COLUMNS) - core_presence.sum(axis=1)
    ).astype("float32")

    result = pd.concat(
        [
            result,
            build_status_n_features(frame),
            build_status_i_features(frame),
            build_status_mr_features(frame),
            build_status_v_features(frame),
        ],
        axis=1,
    )

    # The duplicate-rule group is intentionally categorical.
    for rule_categorical in ("i_duplicate_rule_group",):
        if rule_categorical in result:
            result[rule_categorical] = _clean_category(result[rule_categorical])
            if rule_categorical not in categorical:
                categorical.append(rule_categorical)

    if result.columns.duplicated().any():
        duplicates = result.columns[result.columns.duplicated()].tolist()
        raise RuntimeError(f"Duplicate feature names: {duplicates}")

    if expected_columns is not None:
        expected_categorical_set = set(expected_categorical or [])
        for column in expected_columns:
            if column not in result:
                result[column] = (
                    MISSING_CATEGORY if column in expected_categorical_set else np.nan
                )
        result = result[expected_columns]
        categorical = [
            column for column in expected_columns if column in expected_categorical_set
        ]
    else:
        categorical = [column for column in categorical if column in result]

    if result.empty:
        raise RuntimeError("No model features could be built from the supplied columns.")

    return FeatureResult(
        frame=result,
        categorical_columns=categorical,
        text_columns=[],
    )


## 7. Build aligned training and validation matrices

In [7]:
from sklearn.preprocessing import LabelEncoder

# Keep status separately as the correct answer (y).
y_train_status = normalize_status(train_df["status"])
y_validation_status = normalize_status(validation_df["status"])

# Remove status before feature engineering. The model never receives status.
X_train_raw = train_df.drop(
    columns=["status", "target", "is_non_valid"],
    errors="ignore",
).copy()
X_validation_raw = validation_df.drop(
    columns=["status", "target", "is_non_valid"],
    errors="ignore",
).copy()

train_features = build_features(X_train_raw)

# Remove columns that cannot teach the model anything in this training split.
# Validation and test are then aligned to the retained training columns.
UNINFORMATIVE_TRAIN_FEATURES = [
    column
    for column in train_features.frame.columns
    if train_features.frame[column].nunique(dropna=False) <= 1
]
if UNINFORMATIVE_TRAIN_FEATURES:
    train_features = FeatureResult(
        frame=train_features.frame.drop(columns=UNINFORMATIVE_TRAIN_FEATURES),
        categorical_columns=[
            column for column in train_features.categorical_columns
            if column not in UNINFORMATIVE_TRAIN_FEATURES
        ],
        text_columns=[
            column for column in train_features.text_columns
            if column not in UNINFORMATIVE_TRAIN_FEATURES
        ],
    )

validation_features = build_features(
    X_validation_raw,
    expected_columns=list(train_features.frame.columns),
    expected_categorical=train_features.categorical_columns,
    expected_text=train_features.text_columns,
)

FEATURE_COLUMNS = list(train_features.frame.columns)
CATEGORICAL_COLUMNS = list(train_features.categorical_columns)
TEXT_COLUMNS = list(train_features.text_columns)

status_encoder = LabelEncoder()
y_train_encoded = status_encoder.fit_transform(y_train_status)
STATUS_CLASSES = status_encoder.classes_.astype(str).tolist()

missing_training_classes = sorted(set(VALID_STATUSES) - set(STATUS_CLASSES))
if missing_training_classes:
    raise ValueError(
        "Training data is missing required status classes: "
        + ", ".join(missing_training_classes)
    )

validation_seen_mask = y_validation_status.isin(STATUS_CLASSES).to_numpy()
unseen_validation_classes = sorted(
    set(y_validation_status) - set(STATUS_CLASSES)
)

print("Feature count:", len(FEATURE_COLUMNS))
print("Categorical features:", len(CATEGORICAL_COLUMNS))
print("Text features:", len(TEXT_COLUMNS))
print("Training classes:", STATUS_CLASSES)
print("Unseen validation classes:", unseen_validation_classes)
print("Status present in training features:", "status" in train_features.frame.columns)
print("Status present in validation features:", "status" in validation_features.frame.columns)


print("Automatically removed constant/all-missing features:", UNINFORMATIVE_TRAIN_FEATURES)
rule_feature_columns = [
    column for column in FEATURE_COLUMNS
    if column.startswith(("n_", "i_", "mr_", "v_"))
]
print("Verified N/I/MR/V rule feature count:", len(rule_feature_columns))


Feature count: 72
Categorical features: 7
Text features: 0
Training classes: ['I', 'MR', 'N', 'V']
Unseen validation classes: []
Status present in training features: False
Status present in validation features: False
Automatically removed constant/all-missing features: ['i_prior_missing_relationship_count', 'i_appendix_required_from_date_missing', 'i_appendix_required_to_date_missing', 'i_appendix_required_rate_missing', 'i_appendix_required_tjra_missing', 'i_appendix_required_lra_missing', 'i_appendix_required_ref_do2_no_missing', 'i_appendix_j_rule_found', 'i_appendix_required_missing_count', 'i_rule_history_cutoff_missing', 'mr_entry_flag', 'mr_appendix_value_combination_invalid']
Verified N/I/MR/V rule feature count: 47


In [8]:
# Show every final model feature and how CatBoost will receive it.
raw_input_columns = set(X_train_raw.columns)

feature_inventory = pd.DataFrame({
    "feature_number": range(1, len(FEATURE_COLUMNS) + 1),
    "feature": FEATURE_COLUMNS,
    "origin": [
        "direct/raw" if column in raw_input_columns else "engineered"
        for column in FEATURE_COLUMNS
    ],
    "model_type": [
        "categorical" if column in CATEGORICAL_COLUMNS
        else "text" if column in TEXT_COLUMNS
        else "numeric"
        for column in FEATURE_COLUMNS
    ],
})

display(feature_inventory)
print("Final feature count:", len(FEATURE_COLUMNS))
print("Features intentionally removed:", sorted(FEATURES_TO_DROP))

print("Automatically removed as uninformative:", UNINFORMATIVE_TRAIN_FEATURES)


,feature_number,feature,origin,model_type
0,1,do2_group,direct/raw,categorical
1,2,do2_type,direct/raw,categorical
2,3,description,direct/raw,categorical
3,4,transcription_type,direct/raw,categorical
4,5,delayed,direct/raw,categorical
...,...,...,...,...
67,68,i_less_than_six_months_service,engineered,numeric
68,69,i_less_than_eight_years_service,engineered,numeric
69,70,v_appt_nonactionable_after_7cpc,engineered,numeric
70,71,v_verified_static_violation_count,engineered,numeric


Final feature count: 72
Features intentionally removed: ['abs_duration_days', 'abs_submission_delay_days', 'army_no_is_placeholder', 'check_digit', 'conditional_required_missing_count', 'conditional_required_present_count', 'core_required_present_count', 'created_at_plausible', 'created_before_do2', 'description_char_count', 'description_do2_type', 'description_has_digit', 'description_word_count', 'do2_date_plausible', 'do2_group_type', 'do2_item_no', 'from_date_plausible', 'has_cfa_sanction_date', 'has_cfa_sanction_letter_no', 'has_cfa_sanction_received', 'has_column9_remarks', 'has_pto_remarks', 'has_punishment', 'log_abs_duration_days', 'log_abs_submission_delay_days', 'long_duration', 'lra_is_negative', 'lra_log_abs', 'prior_army_do2_type_row_count', 'prior_batch_do2_item_share', 'prior_batch_row_count', 'prior_do2_no_row_count', 'prior_unit_do2_type_row_count', 'punishment', 'qe', 'rate_is_negative', 'rate_log_abs', 'rate_lra_abs_diff', 'rate_tjra_abs_diff', 'same_day_created_do2

## 8. Train CatBoost with training-period prior correction

In [9]:
import pandas as pd
from pandas.api.types import is_object_dtype, is_string_dtype
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import f1_score


def is_string_like(series):
    return (
        is_object_dtype(series.dtype)
        or is_string_dtype(series.dtype)
        or isinstance(series.dtype, pd.CategoricalDtype)
    )


# Detect categories from both training and validation, then freeze the list.
detected_categorical = [
    column
    for column in train_features.frame.columns
    if (
        is_string_like(train_features.frame[column])
        or is_string_like(validation_features.frame[column])
    )
]
CATEGORICAL_COLUMNS = [
    column
    for column in FEATURE_COLUMNS
    if column in set(CATEGORICAL_COLUMNS) | set(detected_categorical)
]
TEXT_COLUMNS = []
NUMERIC_COLUMNS = [
    column for column in FEATURE_COLUMNS if column not in CATEGORICAL_COLUMNS
]


def prepare_model_frame(feature_frame):
    prepared = feature_frame.reindex(columns=FEATURE_COLUMNS).copy()
    for column in CATEGORICAL_COLUMNS:
        prepared[column] = (
            prepared[column]
            .astype("string")
            .fillna(MISSING_CATEGORY)
            .str.strip()
            .replace("", MISSING_CATEGORY)
            .astype(str)
        )
    for column in NUMERIC_COLUMNS:
        prepared[column] = pd.to_numeric(prepared[column], errors="coerce")
    return prepared


X_train_model = prepare_model_frame(train_features.frame)
X_validation_model = prepare_model_frame(validation_features.frame)
y_validation_encoded = status_encoder.transform(y_validation_status)

if list(X_train_model.columns) != list(X_validation_model.columns):
    raise ValueError("Training and validation feature columns do not match")

print("Training shape:", X_train_model.shape)
print("Validation shape:", X_validation_model.shape)
print("Categorical feature count:", len(CATEGORICAL_COLUMNS))
print("Numeric feature count:", len(NUMERIC_COLUMNS))
print("Status in model input:", "status" in X_train_model.columns)

# Correct the deliberately sampled training distribution using only the natural
# class distribution from the training period—not validation or test labels.
sampled_training_prior = (
    y_train_status.value_counts(normalize=True)
    .reindex(STATUS_CLASSES)
    .fillna(0.0)
)
expected_real_prior = (
    training_population_prior.set_index("status")["rows"]
    .reindex(STATUS_CLASSES)
    .fillna(0.0)
    .astype(float)
)
expected_real_prior = expected_real_prior / expected_real_prior.sum()

if sampled_training_prior.eq(0).any():
    missing = sampled_training_prior[sampled_training_prior.eq(0)].index.tolist()
    raise ValueError(f"Sampled training data is missing classes: {missing}")

def make_train_sample_weight(power):
    prior_ratio = expected_real_prior / sampled_training_prior
    weight_by_status = prior_ratio ** power
    weight = y_train_status.map(weight_by_status).astype(float).to_numpy()
    return weight / weight.mean(), weight_by_status


def equal_weight_monthly_macro_f1(actual, predicted, timestamps):
    scored = pd.DataFrame({
        "actual": np.asarray(actual),
        "predicted": np.asarray(predicted),
        "month": parse_datetime_mixed(pd.Series(timestamps)).dt.to_period("M").astype(str),
    })
    rows = []
    for month, group in scored.groupby("month", dropna=False):
        rows.append({
            "month": str(month),
            "macro_f1": float(f1_score(
                group["actual"], group["predicted"], labels=STATUS_CLASSES,
                average="macro", zero_division=0,
            )),
        })
    monthly = pd.DataFrame(rows)
    return float(monthly["macro_f1"].mean()), monthly


# Validation remains unweighted throughout -- it is only ever used for early
# stopping and metric comparison, never for training weights.
validation_pool = Pool(
    data=X_validation_model,
    label=y_validation_encoded,
    cat_features=CATEGORICAL_COLUMNS,
)

BASE_MODEL_PARAMETERS = {
    "learning_rate": LEARNING_RATE,
    "depth": DEPTH,
    "l2_leaf_reg": 10,
    "min_data_in_leaf": 100,
    "loss_function": "MultiClass",
    # Use multiclass log loss for stable early stopping.
    "eval_metric": "MultiClass",
    # These are reported but not used for early stopping.
    "custom_metric": [
        "Accuracy",
        "TotalF1:average=Macro",
        "TotalF1:average=Weighted",
    ],
    # Do not use auto_class_weights -- reweighting is handled explicitly via
    # train_sample_weight below, driven by the sweep against real class prior.
    "random_seed": RANDOM_SEED,
    "task_type": TASK_TYPE,
    "allow_writing_files": False,
    "verbose": False,
    "od_type": "Iter",
}
if TASK_TYPE == "GPU":
    BASE_MODEL_PARAMETERS["devices"] = DEVICES

# ---------------------------------------------------------
# 6. Sweep PRIOR_CORRECTION_POWER against validation macro-F1
#
# A single fixed correction power was previously chosen without evidence.
# Try each candidate with a fast, capped-iteration model (still using the
# same early stopping and validation pool) and keep the power that gives
# the best macro-F1 across all six classes on validation -- not accuracy,
# which is dominated by V and would just pick whichever power predicts V
# most often.
# ---------------------------------------------------------

SWEEP_ITERATIONS = min(ITERATIONS, 600)
sweep_rows = []
sweep_models = {}

for power in PRIOR_CORRECTION_POWER_GRID:
    weight, weight_by_status = make_train_sample_weight(power)
    sweep_pool = Pool(
        data=X_train_model,
        label=y_train_encoded,
        cat_features=CATEGORICAL_COLUMNS,
        weight=weight,
    )
    sweep_parameters = dict(BASE_MODEL_PARAMETERS, iterations=SWEEP_ITERATIONS)
    sweep_model = CatBoostClassifier(**sweep_parameters)
    sweep_model.fit(
        sweep_pool,
        eval_set=validation_pool,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        use_best_model=True,
    )
    sweep_probabilities = sweep_model.predict_proba(X_validation_model)
    sweep_predicted = status_encoder.inverse_transform(
        sweep_probabilities.argmax(axis=1)
    )
    sweep_macro_f1 = f1_score(
        y_validation_status,
        sweep_predicted,
        labels=STATUS_CLASSES,
        average="macro",
        zero_division=0,
    )
    sweep_monthly_macro_f1, _ = equal_weight_monthly_macro_f1(
        y_validation_status,
        sweep_predicted,
        validation_df[SPLIT_DATE_COLUMN],
    )
    sweep_accuracy = float((sweep_predicted == y_validation_status).mean())
    sweep_rows.append({
        "prior_correction_power": power,
        "validation_macro_f1": sweep_macro_f1,
        "validation_equal_weight_monthly_macro_f1": sweep_monthly_macro_f1,
        "validation_accuracy": sweep_accuracy,
        "best_iteration": sweep_model.get_best_iteration(),
    })
    sweep_models[power] = sweep_model
    print(
        f"power={power:.2f}  macro_f1={sweep_macro_f1:.4f}  "
        f"monthly_macro_f1={sweep_monthly_macro_f1:.4f}  "
        f"accuracy={sweep_accuracy:.4f}  best_iter={sweep_model.get_best_iteration()}"
    )

prior_correction_sweep = pd.DataFrame(sweep_rows).sort_values(
    ["validation_equal_weight_monthly_macro_f1", "validation_macro_f1"],
    ascending=False,
).reset_index(drop=True)
display(prior_correction_sweep)

BEST_PRIOR_CORRECTION_POWER = float(
    prior_correction_sweep.iloc[0]["prior_correction_power"]
)
print("Selected PRIOR_CORRECTION_POWER:", BEST_PRIOR_CORRECTION_POWER)

# Free the sweep models (only the selected power is retrained at full
# iterations below) to avoid holding several CatBoost models in memory.
del sweep_models
import gc as _gc
_ = _gc.collect()

train_sample_weight, sampling_weight_by_status = make_train_sample_weight(
    BEST_PRIOR_CORRECTION_POWER
)
sampling_weight_report = pd.DataFrame({
    "sampled_training_prior": sampled_training_prior,
    "natural_training_prior": expected_real_prior,
    "training_weight": sampling_weight_by_status,
})
display(sampling_weight_report)

# ---------------------------------------------------------
# 7. Create weighted CatBoost Pool objects for the final model
# ---------------------------------------------------------

train_pool = Pool(
    data=X_train_model,
    label=y_train_encoded,
    cat_features=CATEGORICAL_COLUMNS,
    weight=train_sample_weight,
)

# ---------------------------------------------------------
# 8. Train the final multiclass model at full iterations, using the
#    sweep-selected reweighting.
# ---------------------------------------------------------

model_parameters = dict(BASE_MODEL_PARAMETERS, iterations=ITERATIONS, verbose=100)
model = CatBoostClassifier(**model_parameters)

model.fit(
    train_pool,
    eval_set=validation_pool,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    use_best_model=True,
)

print("Best iteration:", model.get_best_iteration())
print("Model classes:", STATUS_CLASSES)
print("Class weights:", model.get_param("auto_class_weights"))
print("Final PRIOR_CORRECTION_POWER used:", BEST_PRIOR_CORRECTION_POWER)


# ---------------------------------------------------------
# 9. Bounded feature-family ablation on the validation period.
# Selection uses equal-weight monthly macro-F1. The production model above
# still uses all verified features; this table tells us what to remove next.
# ---------------------------------------------------------
feature_group_ablation = pd.DataFrame()
if RUN_FEATURE_GROUP_ABLATION:
    rule_prefixes = ("n_", "i_", "mr_", "v_")
    base_columns = [c for c in FEATURE_COLUMNS if not c.startswith(rule_prefixes)]
    ablation_groups = {
        "BASE": base_columns,
        "BASE+N": base_columns + [c for c in FEATURE_COLUMNS if c.startswith("n_")],
        "BASE+I": base_columns + [c for c in FEATURE_COLUMNS if c.startswith("i_")],
        "BASE+MR": base_columns + [c for c in FEATURE_COLUMNS if c.startswith("mr_")],
        "BASE+V": base_columns + [c for c in FEATURE_COLUMNS if c.startswith("v_")],
        "ALL_VERIFIED": FEATURE_COLUMNS,
    }
    ablation_rows = []
    for group_name, group_columns in ablation_groups.items():
        group_categorical = [c for c in CATEGORICAL_COLUMNS if c in group_columns]
        group_train_pool = Pool(
            X_train_model[group_columns], y_train_encoded,
            cat_features=group_categorical, weight=train_sample_weight,
        )
        group_validation_pool = Pool(
            X_validation_model[group_columns], y_validation_encoded,
            cat_features=group_categorical,
        )
        group_model = CatBoostClassifier(
            **dict(BASE_MODEL_PARAMETERS, iterations=min(ITERATIONS, ABLATION_ITERATIONS))
        )
        group_model.fit(
            group_train_pool,
            eval_set=group_validation_pool,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            use_best_model=True,
        )
        group_predicted = status_encoder.inverse_transform(
            group_model.predict_proba(X_validation_model[group_columns]).argmax(axis=1)
        )
        group_macro = f1_score(
            y_validation_status, group_predicted, labels=STATUS_CLASSES,
            average="macro", zero_division=0,
        )
        group_monthly, _ = equal_weight_monthly_macro_f1(
            y_validation_status, group_predicted, validation_df[SPLIT_DATE_COLUMN]
        )
        ablation_rows.append({
            "feature_group": group_name,
            "feature_count": len(group_columns),
            "validation_macro_f1": float(group_macro),
            "validation_equal_weight_monthly_macro_f1": group_monthly,
            "best_iteration": group_model.get_best_iteration(),
        })
        del group_model, group_train_pool, group_validation_pool
    feature_group_ablation = pd.DataFrame(ablation_rows).sort_values(
        ["validation_equal_weight_monthly_macro_f1", "validation_macro_f1"],
        ascending=False,
    ).reset_index(drop=True)
    display(feature_group_ablation)


Training shape: (372265, 72)
Validation shape: (237745, 72)
Categorical feature count: 7
Numeric feature count: 65
Status in model input: False
power=0.00  macro_f1=0.6943  monthly_macro_f1=0.5674  accuracy=0.8925  best_iter=142
power=0.25  macro_f1=0.7146  monthly_macro_f1=0.6142  accuracy=0.9004  best_iter=111
power=0.50  macro_f1=0.6089  monthly_macro_f1=0.4978  accuracy=0.9014  best_iter=98
power=0.75  macro_f1=0.6061  monthly_macro_f1=0.5043  accuracy=0.9044  best_iter=91
power=1.00  macro_f1=0.4152  monthly_macro_f1=0.4592  accuracy=0.7919  best_iter=93


,prior_correction_power,validation_macro_f1,validation_equal_weight_monthly_macro_f1,validation_accuracy,best_iteration
0,0.25,0.714638,0.614213,0.900414,111
1,0.00,0.694294,0.567354,0.892528,142
2,0.75,0.606141,0.504279,0.904427,91
3,0.50,0.608926,0.497846,0.901361,98
4,1.00,0.415178,0.459203,0.791928,93


Selected PRIOR_CORRECTION_POWER: 0.25


,sampled_training_prior,natural_training_prior,training_weight
status,,,
I,0.222425,0.037091,0.639031
MR,0.086981,0.014505,0.639031
N,0.019029,0.003173,0.639031
V,0.671565,0.945231,1.089212


0:	learn: 1.3273819	test: 1.3271544	best: 1.3271544 (0)	total: 19.9ms	remaining: 39.7s
100:	learn: 0.3578360	test: 0.4578443	best: 0.4576166 (97)	total: 1.74s	remaining: 32.6s
200:	learn: 0.2905600	test: 0.4834835	best: 0.4528003 (111)	total: 3.43s	remaining: 30.7s
bestTest = 0.4528003349
bestIteration = 111
Shrink model to first 112 iterations.
Best iteration: 111
Model classes: ['I', 'MR', 'N', 'V']
Class weights: None
Final PRIOR_CORRECTION_POWER used: 0.25


,feature_group,feature_count,validation_macro_f1,validation_equal_weight_monthly_macro_f1,best_iteration
0,ALL_VERIFIED,72,0.714638,0.614213,111
1,BASE+N,43,0.694306,0.567888,309
2,BASE+I,51,0.606888,0.467088,120
3,BASE+V,28,0.579754,0.443277,126
4,BASE,25,0.522955,0.424650,144
5,BASE+MR,25,0.522955,0.424650,144


## 9. Evaluate validation with honest baselines

In [10]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
)


def evaluate_split(split_name, raw_frame, feature_frame, y_status):
    probabilities = model.predict_proba(feature_frame)
    predicted = status_encoder.inverse_transform(probabilities.argmax(axis=1))

    seen_mask = y_status.isin(STATUS_CLASSES).to_numpy()
    present_labels = sorted(set(y_status) & set(STATUS_CLASSES))
    encoded_seen = status_encoder.transform(y_status.loc[seen_mask])

    actual_non_v = y_status.ne("V").to_numpy()
    predicted_non_v = predicted != "V"
    all_v_accuracy = float(y_status.eq("V").mean())

    metrics = {
        "split": split_name,
        "rows": int(len(y_status)),
        "accuracy": float(accuracy_score(y_status, predicted)),
        "all_v_accuracy_baseline": all_v_accuracy,
        "balanced_accuracy": float(balanced_accuracy_score(y_status, predicted)),
        "macro_f1_present_classes": float(
            f1_score(
                y_status,
                predicted,
                labels=present_labels,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_f1_all_model_classes": float(
            f1_score(
                y_status,
                predicted,
                labels=STATUS_CLASSES,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(y_status, predicted, average="weighted", zero_division=0)
        ),
        "non_v_precision": float(
            precision_score(actual_non_v, predicted_non_v, zero_division=0)
        ),
        "non_v_recall": float(
            recall_score(actual_non_v, predicted_non_v, zero_division=0)
        ),
        "review_rate": float(predicted_non_v.mean()),
        "false_negative_non_v": int((actual_non_v & ~predicted_non_v).sum()),
        "multiclass_log_loss": float(
            log_loss(
                encoded_seen,
                probabilities[seen_mask],
                labels=np.arange(len(STATUS_CLASSES)),
            )
        ),
    }

    report = pd.DataFrame(
        classification_report(
            y_status,
            predicted,
            labels=STATUS_CLASSES,
            output_dict=True,
            zero_division=0,
        )
    ).transpose()

    confusion = pd.DataFrame(
        confusion_matrix(y_status, predicted, labels=STATUS_CLASSES),
        index=pd.Index(STATUS_CLASSES, name="actual_status"),
        columns=pd.Index(STATUS_CLASSES, name="predicted_status"),
    )

    identity_columns = [
        column
        for column in (
            "id",
            "created_at",
            "do2_date",
            "do2_type",
            "do2_group",
            "unit_code",
        )
        if column in raw_frame.columns
    ]
    predictions = raw_frame[identity_columns].copy()
    predictions["created_month"] = parse_datetime_mixed(
        raw_frame[SPLIT_DATE_COLUMN]
    ).dt.to_period("M").astype(str).to_numpy()
    transcription = _normalized_text(raw_frame, "transcription_type")
    predictions["source_segment"] = np.where(
        transcription.eq("S"), "SYSTEM_GENERATED", "DIRECT_OR_OTHER"
    )
    delay_days = (
        parse_datetime_mixed(raw_frame[SPLIT_DATE_COLUMN]).dt.normalize()
        - parse_datetime_mixed(raw_frame["do2_date"]).dt.normalize()
    ).dt.days
    predictions["delay_bucket"] = pd.cut(
        delay_days,
        bins=[-np.inf, -1, 30, 90, 365, np.inf],
        labels=["FUTURE_DO2_DATE", "D00_030", "D031_090", "D091_365", "D366_PLUS"],
    ).astype("string").fillna("MISSING").to_numpy()
    predictions["actual_status"] = y_status.to_numpy()
    predictions["predicted_status"] = predicted
    predictions["predicted_status_probability"] = probabilities.max(axis=1)
    predictions["is_correct"] = predictions["actual_status"].eq(
        predictions["predicted_status"]
    )
    for class_index, status_class in enumerate(STATUS_CLASSES):
        predictions[f"probability_status_{status_class}"] = probabilities[:, class_index]

    monthly_macro, _ = equal_weight_monthly_macro_f1(
        y_status, predicted, raw_frame[SPLIT_DATE_COLUMN]
    )
    metrics["equal_weight_monthly_macro_f1"] = monthly_macro
    return metrics, report, confusion, predictions


(
    validation_metrics,
    validation_report,
    validation_confusion,
    validation_predictions,
) = evaluate_split(
    "validation",
    validation_df,
    X_validation_model,
    y_validation_status,
)

display(pd.DataFrame([validation_metrics]))
display(validation_report)
display(validation_confusion)
display(validation_predictions.head())


def build_segment_metrics(predictions):
    rows = []
    for dimension in ("created_month", "source_segment", "delay_bucket"):
        for value, group in predictions.groupby(dimension, dropna=False):
            actual = group["actual_status"]
            predicted = group["predicted_status"]
            rows.append({
                "dimension": dimension,
                "segment": str(value),
                "rows": int(len(group)),
                "accuracy": float(accuracy_score(actual, predicted)),
                "macro_f1": float(f1_score(
                    actual, predicted, labels=STATUS_CLASSES,
                    average="macro", zero_division=0,
                )),
                "review_rate": float(predicted.ne("V").mean()),
            })
    return pd.DataFrame(rows)


validation_segment_metrics = build_segment_metrics(validation_predictions)
display(validation_segment_metrics)


,split,rows,accuracy,all_v_accuracy_baseline,balanced_accuracy,macro_f1_present_classes,macro_f1_all_model_classes,weighted_f1,non_v_precision,non_v_recall,review_rate,false_negative_non_v,multiclass_log_loss,equal_weight_monthly_macro_f1
0,validation,237745,0.900414,0.789447,0.6657,0.714638,0.714638,0.89156,0.92207,0.580986,0.132667,20975,0.4528,0.614213


,precision,recall,f1-score,support
I,0.931094,0.577932,0.713187,48305.000000
MR,0.476018,0.468806,0.472384,1122.000000
N,0.876380,0.629160,0.732472,631.000000
V,0.898280,0.986904,0.940509,187687.000000
accuracy,0.900414,0.900414,0.900414,0.900414
macro avg,0.795443,0.665700,0.714638,237745.000000
weighted avg,0.902897,0.900414,0.891560,237745.000000


predicted_status,I,MR,N,V
actual_status,,,,
I,27917,77,0,20311
MR,24,526,3,569
N,138,1,397,95
V,1904,501,53,185229


,id,created_at,do2_date,do2_type,do2_group,unit_code,created_month,source_segment,delay_bucket,actual_status,predicted_status,predicted_status_probability,is_correct,probability_status_I,probability_status_MR,probability_status_N,probability_status_V
0,14286673,2025-01-15 17:36:31.769,2018-09-25,0,IA,64B014,2025-01,SYSTEM_GENERATED,D366_PLUS,I,I,0.713938,True,0.713938,0.057160,0.036555,0.192347
1,15373818,2025-01-06 18:08:31.402,2024-12-25,0,IA,92B005,2025-01,DIRECT_OR_OTHER,D00_030,V,V,0.794875,True,0.104907,0.073476,0.026743,0.794875
2,15369314,2025-01-09 11:15:15.561,2024-12-16,0,IA,64B015,2025-01,DIRECT_OR_OTHER,D00_030,V,V,0.909101,True,0.048637,0.024807,0.017455,0.909101
3,15485819,2025-03-05 12:18:05.809,2025-01-08,0,IA,HWS166,2025-03,DIRECT_OR_OTHER,D031_090,V,V,0.796727,True,0.067088,0.065479,0.070707,0.796727
4,15477770,2025-03-10 11:16:28.717,2025-02-15,0,IA,92B061,2025-03,DIRECT_OR_OTHER,D00_030,V,V,0.814590,True,0.061953,0.058164,0.065293,0.814590


,dimension,segment,rows,accuracy,macro_f1,review_rate
0,created_month,2025-01,64664,0.897903,0.715838,0.437415
1,created_month,2025-02,29545,0.972483,0.689840,0.031207
2,created_month,2025-03,24666,0.965986,0.540282,0.025298
3,created_month,2025-04,42592,0.982837,0.629201,0.016153
4,created_month,2025-05,23994,0.976286,0.587250,0.023131
5,created_month,2025-06,52284,0.729898,0.522868,0.008932
6,source_segment,DIRECT_OR_OTHER,153294,0.978290,0.680688,0.020927
7,source_segment,SYSTEM_GENERATED,84451,0.759056,0.533256,0.335496
8,delay_bucket,D00_030,179074,0.881803,0.566653,0.010995
9,delay_bucket,D031_090,23327,0.975222,0.703018,0.026536


In [11]:
# Preserve compact training information, then free large matrices before test.
TRAINING_ROWS = int(len(train_df))
VALIDATION_ROWS = int(len(validation_df))
TRAINING_CLASS_COUNTS = {
    str(key): int(value)
    for key, value in y_train_status.value_counts().sort_index().to_dict().items()
}

del train_features, validation_features, train_df, validation_df
del X_train_model, X_validation_model, train_pool, validation_pool
_ = gc.collect()


## 10. Evaluate the untouched 2025 test split

Do not change features, weights, thresholds, or model parameters after reading
this section. A change requires a new final holdout period.

In [12]:
test_raw = load_or_cache_enriched(
    "test",
    TEST_START,
    TEST_END,
    MAX_TEST_ROWS,
    lambda: load_holdout_data(TEST_START, TEST_END, MAX_TEST_ROWS),
)
test_df, test_cleaning_report = clean_split(
    test_raw, "test", drop_incomplete_v_labels=False
)
del test_raw
_ = gc.collect()

y_test_status = normalize_status(test_df["status"])

# Keep y_test_status as the answer and remove status from model inputs.
X_test_raw = test_df.drop(
    columns=["status", "target", "is_non_valid"],
    errors="ignore",
).copy()
test_features = build_features(
    X_test_raw,
    expected_columns=FEATURE_COLUMNS,
    expected_categorical=CATEGORICAL_COLUMNS,
    expected_text=TEXT_COLUMNS,
)

unseen_test_classes = sorted(set(y_test_status) - set(STATUS_CLASSES))
if unseen_test_classes:
    raise ValueError(
        "Test contains statuses the model cannot predict: "
        + ", ".join(unseen_test_classes)
    )

print("Status present in test features:", "status" in test_features.frame.columns)

test_split_check = validate_split(test_df, "test", TEST_START, TEST_END)
TEST_IDS = frozenset(test_df["id"].tolist())
train_test_overlap = len(TRAIN_IDS & TEST_IDS)
validation_test_overlap = len(VALIDATION_IDS & TEST_IDS)
if train_test_overlap or validation_test_overlap:
    raise ValueError(
        "Split id overlap detected: "
        f"train/test={train_test_overlap:,}, "
        f"validation/test={validation_test_overlap:,}"
    )

X_test_model = prepare_model_frame(test_features.frame)

test_metrics, test_report, test_confusion, test_predictions = evaluate_split(
    "test",
    test_df,
    X_test_model,
    y_test_status,
)

display(pd.DataFrame([test_cleaning_report]))
display(pd.DataFrame([test_split_check]))
print("Train/test id overlap:", train_test_overlap)
print("Validation/test id overlap:", validation_test_overlap)
display(split_profile(test_df, "test"))
display(pd.DataFrame([test_metrics]))
display(test_report)
display(test_confusion)
display(test_predictions.head())


test_segment_metrics = build_segment_metrics(test_predictions)
display(test_segment_metrics)


Holdout date columns loaded as text: ['do2_date', 'from_date', 'to_date', 'created_at']
Holdout rows enriched with status-independent prior candidate facts.
Database enrichment for test finished in 7.9 minutes
Cached enriched test data: /home/ruthikreddy/Desktop/do2/do2_status_prediction/data/model_cache/time_safe_v6_status_independent_history_test_2025-07-01_2026-01-01_250000.parquet
Status present in test features: False


,split,initial_rows,unparseable_created_at_rows,missing_created_at_rows,sentinel_created_at_rows,malformed_created_at_rows,excluded_status_rows,excluded_lifecycle_status_rows,excluded_do2_group_rows,incomplete_v_rows_found,incomplete_v_rows_removed,final_rows
0,test,250000,0,0,0,0,13310,13310,0,0,0,236690


,split,rows,unique_ids,minimum_created_at,maximum_created_at
0,test,236690,236690,2025-07-01 10:27:57.447,2025-12-30 01:52:25.482837


Train/test id overlap: 0
Validation/test id overlap: 0


,split,status,rows,percentage
0,test,V,231658,97.874012
1,test,I,3954,1.67054
2,test,N,559,0.236174
3,test,MR,519,0.219274


,split,rows,accuracy,all_v_accuracy_baseline,balanced_accuracy,macro_f1_present_classes,macro_f1_all_model_classes,weighted_f1,non_v_precision,non_v_recall,review_rate,false_negative_non_v,multiclass_log_loss,equal_weight_monthly_macro_f1
0,test,236690,0.942887,0.97874,0.53135,0.472837,0.472837,0.954675,0.160405,0.364865,0.048359,3196,0.244997,0.452687


,precision,recall,f1-score,support
I,0.060818,0.142893,0.085322,3954.000000
MR,0.111538,0.335260,0.167388,519.000000
N,0.645973,0.688730,0.666667,559.000000
V,0.985811,0.958516,0.971972,231658.000000
accuracy,0.942887,0.942887,0.942887,0.942887
macro avg,0.451035,0.531350,0.472837,236690.000000
weighted avg,0.967639,0.942887,0.954675,236690.000000


predicted_status,I,MR,N,V
actual_status,,,,
I,565,686,17,2686
MR,1,174,6,338
N,2,0,385,172
V,8722,700,188,222048


,id,created_at,do2_date,do2_type,do2_group,unit_code,created_month,source_segment,delay_bucket,actual_status,predicted_status,predicted_status_probability,is_correct,probability_status_I,probability_status_MR,probability_status_N,probability_status_V
0,15854361,2025-09-23 14:22:26.323,2025-09-05,0,IA,64B006,2025-09,DIRECT_OR_OTHER,D00_030,V,V,0.943121,True,0.025137,0.019637,0.012106,0.943121
1,15902007,2025-10-15 17:32:31.899,2025-10-15,0,IA,64B022,2025-10,SYSTEM_GENERATED,D00_030,V,V,0.949487,True,0.026909,0.013836,0.009768,0.949487
2,15736312,2025-07-16 10:55:01.158,2025-07-01,0,IA,55C091,2025-07,SYSTEM_GENERATED,D00_030,V,I,0.599334,False,0.599334,0.021166,0.011662,0.367838
3,15885538,2025-10-13 16:06:51.732,2025-09-09,0,IA,RCB041,2025-10,DIRECT_OR_OTHER,D031_090,V,V,0.855356,True,0.076636,0.046778,0.021230,0.855356
4,15838051,2025-09-17 15:13:30.437,2025-08-24,0,IA,64B005,2025-09,DIRECT_OR_OTHER,D00_030,V,V,0.917704,True,0.036161,0.023904,0.022231,0.917704


,dimension,segment,rows,accuracy,macro_f1,review_rate
0,created_month,2025-07,46092,0.828972,0.353224,0.160874
1,created_month,2025-08,39681,0.974597,0.516651,0.021723
2,created_month,2025-09,34537,0.961954,0.552312,0.028202
3,created_month,2025-10,61758,0.980828,0.377067,0.011723
4,created_month,2025-11,22917,0.964393,0.435389,0.022996
5,created_month,2025-12,31705,0.958587,0.481480,0.029774
6,source_segment,DIRECT_OR_OTHER,192607,0.971138,0.506341,0.019169
7,source_segment,SYSTEM_GENERATED,44083,0.819454,0.227167,0.175895
8,delay_bucket,D00_030,197179,0.950015,0.490541,0.044442
9,delay_bucket,D031_090,31870,0.959460,0.414769,0.022215


In [13]:
def show_status_distribution(status_series, split_name):
    result = (
        status_series
        .fillna("MISSING")
        .astype(str)
        .str.strip()
        .str.upper()
        .value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="rows")
    )

    result["percentage"] = (
        result["rows"] / result["rows"].sum() * 100
    )

    result.insert(0, "split", split_name)

    return result


status_distribution = pd.concat(
    [
        show_status_distribution(y_train_status, "train"),
        show_status_distribution(y_validation_status, "validation"),
        show_status_distribution(y_test_status, "test"),
    ],
    ignore_index=True,
)

display(status_distribution)


,split,status,rows,percentage
0,train,V,250000,67.156461
1,train,I,82801,22.242489
2,train,MR,32380,8.698105
3,train,N,7084,1.902945
4,validation,V,187687,78.944668
5,validation,I,48305,20.317988
6,validation,MR,1122,0.471934
7,validation,N,631,0.265410
8,test,V,231658,97.874012
9,test,I,3954,1.670540


## 11. Feature importance

In [14]:
feature_importance = (
    pd.DataFrame(
        {
            "feature": FEATURE_COLUMNS,
            "importance": model.get_feature_importance(),
        }
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print(feature_importance)


                                feature  importance
0                           description   13.855572
1                 submission_delay_days   13.013718
2              from_date_after_do2_date    9.376501
3                          has_column_c    8.588176
4               i_reference_match_count    5.568745
..                                  ...         ...
67          i_from_date_within_fs_month    0.000000
68       i_less_than_six_months_service    0.000000
69      i_less_than_eight_years_service    0.000000
70    v_verified_static_violation_count    0.000000
71  v_all_verified_static_checks_passed    0.000000

[72 rows x 2 columns]


## 12. Save the model and reproducibility reports

In [15]:
if not SAVE_ARTIFACTS:
    raise RuntimeError("Set SAVE_ARTIFACTS=True to save the final model")

MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "do2_catboost_multiclass.cbm"
metadata_path = MODEL_DIR / "model_metadata_multiclass.json"
model.save_model(model_path)

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_version": datetime.now(timezone.utc).strftime(
        "do2-catboost-multiclass-%Y%m%dT%H%M%SZ"
    ),
    "model_source": "do2_file(4)_evidence_safe_v2.ipynb",
    "feature_set_version": FEATURE_SET_VERSION,
    "model_type": "multiclass_status",
    "model_target": f"normalized status in {VALID_STATUSES}",
    "target_mode": TARGET_MODE,
    "classes": STATUS_CLASSES,
    "feature_columns": FEATURE_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "text_columns": TEXT_COLUMNS,
    "required_input_columns": REQUIRED_INPUT_COLUMNS,
    "candidate_columns": CANDIDATE_COLUMNS,
    "features_dropped": sorted(FEATURES_TO_DROP),
    "uninformative_training_features_dropped": UNINFORMATIVE_TRAIN_FEATURES,
    "rule_feature_columns": [c for c in FEATURE_COLUMNS if c.startswith(("n_", "i_", "mr_", "v_"))],
    "rule_source_tables": ["do2", "employee", "status_code", "lpc_register", "appendix_j", "family_detail"],
    "data_sources": {
        "train": f"{settings.db_schema}.{TRAIN_TABLE}",
        "validation": f"{settings.db_schema}.{SOURCE_TABLE}",
        "test": f"{settings.db_schema}.{SOURCE_TABLE}",
    },
    "split_date_column": SPLIT_DATE_COLUMN,
    "training_window": [TRAIN_START, TRAIN_END],
    "validation_window": [VALIDATION_START, VALIDATION_END],
    "test_window": [TEST_START, TEST_END],
    "prior_correction_power_grid": list(PRIOR_CORRECTION_POWER_GRID),
    "prior_correction_power_selected": BEST_PRIOR_CORRECTION_POWER,
    "prior_selection_metric": "validation_equal_weight_monthly_macro_f1",
    "created_at_caveat": "Java may reset created_at during pending reprocessing; lower id plus earlier/equal created_at is the conservative history proxy",
    "history_status_policy": "No prior do2.status or processed_date used",
    "appendix_j_historical_features_enabled": USE_CURRENT_APPENDIX_J_FOR_HISTORICAL_ROWS,
    "max_train_v_rows": MAX_TRAIN_V_ROWS,
    "max_validation_rows": MAX_VALIDATION_ROWS,
    "max_test_rows": MAX_TEST_ROWS,
    "training_rows": TRAINING_ROWS,
    "validation_rows": VALIDATION_ROWS,
    "test_rows": int(len(test_df)),
    "split_overlap_counts": {
        "train_validation": train_validation_overlap,
        "train_test": train_test_overlap,
        "validation_test": validation_test_overlap,
    },
    "training_class_counts": TRAINING_CLASS_COUNTS,
    "unseen_validation_classes": unseen_validation_classes,
    "unseen_test_classes": unseen_test_classes,
    "best_iteration": int(model.get_best_iteration()),
    "model_parameters": model.get_params(),
    "dependency_versions": {
        package: importlib.metadata.version(package)
        for package in (
            "catboost",
            "pandas",
            "numpy",
            "scikit-learn",
            "pyarrow",
        )
    },
    "metrics": {
        "validation": validation_metrics,
        "test": test_metrics,
    },
}

with metadata_path.open("w", encoding="utf-8") as handle:
    json.dump(metadata, handle, indent=2)

validation_report.to_csv(MODEL_DIR / "validation_classification_report.csv")
validation_confusion.to_csv(MODEL_DIR / "validation_confusion_matrix.csv")
validation_predictions.to_parquet(
    MODEL_DIR / "validation_predictions_multiclass.parquet",
    index=False,
)

test_report.to_csv(MODEL_DIR / "test_classification_report.csv")
test_confusion.to_csv(MODEL_DIR / "test_confusion_matrix.csv")
test_predictions.to_parquet(
    MODEL_DIR / "test_predictions_multiclass.parquet",
    index=False,
)
feature_importance.to_csv(MODEL_DIR / "feature_importance.csv", index=False)
validation_segment_metrics.to_csv(
    MODEL_DIR / "validation_segment_metrics.csv", index=False
)
test_segment_metrics.to_csv(MODEL_DIR / "test_segment_metrics.csv", index=False)
if not feature_group_ablation.empty:
    feature_group_ablation.to_csv(
        MODEL_DIR / "feature_group_ablation.csv", index=False
    )

print("Model saved to:", model_path)
print("Metadata saved to:", metadata_path)
print("Model version:", metadata["model_version"])

sampling_weight_report.to_csv(
    MODEL_DIR / "training_sampling_weights.csv",
    index=True,
)
prior_correction_sweep.to_csv(
    MODEL_DIR / "prior_correction_power_sweep.csv",
    index=False,
)
training_population_prior.to_csv(
    MODEL_DIR / "natural_training_class_distribution.csv",
    index=False,
)


Model saved to: /home/ruthikreddy/Desktop/do2/do2_status_prediction/models_prac_v4_time_safe/do2_catboost_multiclass.cbm
Metadata saved to: /home/ruthikreddy/Desktop/do2/do2_status_prediction/models_prac_v4_time_safe/model_metadata_multiclass.json
Model version: do2-catboost-multiclass-20260819T084853Z


## Next steps

Compare base-only, +N, +I, and +MR feature groups using rolling temporal
validation before removing features. Add verified C and P rule features next.
Keep review routing separate from exact-status prediction; do not restore the
old `0.11` non-V threshold without probability calibration and multi-period
validation.